# 🩺 RSNA Knee Abnormality Detection: A Diagnostic Quest

Welcome to the clinic of the future!  
You are a radiologist — but not an ordinary one. You are armed with neural networks.  
Your task is to make **12 key diagnoses** from knee MRI studies, using both images and, during training, the accompanying radiology reports.

---

## 🧩 The Core Challenge

Each study is a **puzzle** made of several MRI series — slices acquired in different planes and with different contrast weightings.  
You must assemble the full picture and answer 12 questions: Is there a ligament tear? Meniscal damage? Osteoarthritis? Inflammation? A cyst? A fracture?

**Metric:** macro-averaged ROC AUC across all 12 labels.  
**Key twist:** during training, you have access not only to the images but also to the **radiologists’ free-text reports**. At inference time, reports are absent — so you must learn to squeeze every bit of signal from them while training.

---

## 🧠 Analogy: “Training a Virtual Radiology Resident”

Imagine you are teaching a new resident:

- **DICOM series** — different examination tools: sagittal, coronal, and axial views.
- **Fluid-sensitive sequences** (T2, PD, STIR) — a flashlight that highlights fluid, edema, and tears.
- **train.csv** — patient charts with partially completed diagnoses.
- **train_series.csv** — a description of each scan: plane, fat suppression, fluid sensitivity.
- **Radiology reports** — notes from an experienced mentor who sometimes writes “ACL tear” and sometimes “intact menisci.”

Your resident (the model) must learn to make diagnoses even when the mentor is not there (at test time).

---

## 🏆 Difficulty Tiers of the Diagnoses

| Tier | Diagnoses | Why They Are Hard | What Helps |
|------|-----------|-------------------|------------|
| 🟢 **Easier** | Baker's cyst, Effusion | Clearly visible on fluid-sensitive sequences | A single sagittal series |
| 🟡 **Moderate** | MCL, Medial/Lateral Meniscus, PF OA | Need assessment across multiple planes | 2–3 series + attention |
| 🔴 **Hard** | ACL, Synovitis, Contusion, Fracture | May be subtle; require broader context | Multi-plane aggregation, 2.5D/3D inputs |
| 🟣 **Very Hard** | Medial OA, Lateral OA | Cartilage thickness assessment; need thin slices | Specialized models + post-processing |

---

## 💡 The Main Takeaway

**This competition is won by whoever does three things best:**

1. **Select the right series** — not all 500 GB of data is useful.
2. **Exploit the reports** to expand weak labels — they are free supervision.
3. **Aggregate carefully** across planes and slices.

No magic — only solid data engineering and thoughtful modeling.

---

**Let’s go!** 🚀  
We start with a simple diagnostic tool and gradually turn it into a radiology expert.

# 📊 Evolution of Public Solutions: 0.937 → 0.94 → 0.941 → 0.942 → 0.943 → 0.950

## 🥉 Work 1 — 0.937
**Head and Shoulders, Knees and Toes**

🔗 https://www.kaggle.com/code/prvsiyan/head-and-shoulders-knees-and-toes

- CoAtNet branch expanded to **4 arms**: v5, v10, v5-reverse, v8.
- Reverse-flip adds diversity without new training.
- Outer CoAtNet weight becomes **global 0.60** instead of per-target.

---

## 🥈 Work 2 — 0.94
**RSNA Knee Hybrid Raptor — Lateral Meniscus R100**

🔗 https://www.kaggle.com/code/renta0426/rsna-knee-hybrid-raptor-lateral-meniscus-r100

- Starts from the exact 0.939 parent (reproduced 1-to-1).
- Changes **only one label**: Lateral Meniscus.
- Outer routing for Lateral Meniscus: Transformer/Rad **0.00** / strengthened Raptor **1.00**.
- All other 11 labels remain **bit-for-bit equal** to the parent at 0.40 / 0.60.

> **Key idea:** isolate one problematic target and give it the full weight of the strongest branch.

---

## 🥇 Work 3 — 0.941
**Bend the Knee to the Dinosaurs (ALL PUBLIC)**

🔗 https://www.kaggle.com/code/mattiaangeli/bend-the-knee-to-the-dinosaurs

- Keeps the 4-arm Raptor from Work 1.
- Adds **residual-gated CoAtNet** (e4/e6/e8) inside the Raptor branch.
- Inner blend: public Raptor 0.60 / residual-CoAt 0.40.
- Adds a public **CoAtNet MRI reader**: 384 px adjacent-slice triplets → CoAtNet RMLP-2 backbone.
- Finding-specific gated spatial head + second attention stage pools windows across the study.
- Updated CoAtNet arm weights: maxspan-v5 = **0.60**, reverse = **0.10**.
- Per-target outer routing: ACL **0.75**, Medial Meniscus **0.80**, Lateral Meniscus **1.00**, Lateral OA **0.75**, Fracture **0.75**.

> **Key idea:** strengthen the Raptor arm with a genuinely different visual inductive bias and let each finding lean on whichever branch predicts it best.

---

## 🚀 Work 4 — 0.942
**rsna-knee-0942-restructured**

🔗 https://www.kaggle.com/code/maverickss26/rsna-knee-0942-restructured

### What changed vs 0.941

Three **hyperparameters** raised — no new models:

| Parameter | 0.941 | 0.942 |
|-----------|-------|-------|
| `a5_w` (A5 blend weight) | 0.45 | **0.52** |
| `rad_alpha` (RadImageNet) | 0.50 | **0.55** |
| `rad_second_alpha` (E13 second pass) | 0.15 | **0.20** |
| `raptor_k_eval` (windows per arm) | 62 / 42 | **94** |

### Why it helps

- **`raptor_k_eval=94`** — samples each stack more densely. This is an accuracy change that needed no new model, and it is only affordable because of the speed work: shared DINO prefixes, persistent T4×2 replicas, cached decoding. Full run takes **~4 minutes** on T4×2.
- **Reweighted A5 and RadImageNet** — both branches contribute more to the final blend.

### Added in this version

- **E9 RadImageNet Heads V15 dataset** — an alternative source of heads (`antoinegg1/rsna-knee-e9-radimagenet-heads-v15`).
- **Parallel execution** of residual-gated CoAtNet and D4 DepthZone SWA3 — 155s → 69s with zero change to predictions.
- **Routing diagnostics** — an agreement / tie-block table.
- **Presets:** `speedy`, `parent`, `halfway`, `sparse`.

---

## ⚡ Work 5 — 0.943
**rsna-knee-fast-2xt4-inference**

🔗 https://www.kaggle.com/code/jiweiliu/rsna-knee-fast-2xt4-inference

### What changed vs 0.942

The CoAt family **doubled** — from 2 members to 4 — and the way they are combined was changed.

| Component | 0.942 | 0.943 |
|-----------|-------|-------|
| CoAt family members | 2 (resgated, d4) | **4** (+ global96, + repairv1) |
| Global96 epochs | — | **16 / 23 / 18** |
| Repair-v1 epochs | — | **12 / 7 / 11** |
| Family reduction | mean of ranks | **rank of mean of probabilities** |
| Parallelism | pair | **two pairs in parallel** |
| Raptor header cache | — | **shared from DINOv2 ordering scan** |

### New models in this version

- **Global96 top-3** — same 96-slice stack as D4, but a full-width gated head instead of Rank8 finding-to-slot attention. Shares geometry, not the reader.
- **Repair-v1 top-3** — third independent CoAt checkpoint, adds another 3-checkpoint vote.

### Family reduction change

Previous: rank each member, average ranks, rank the average.
New: read each member's `probability_mean` from its `.npz`, average **probabilities**, then `rank(pct=True)`.
Fallback to the old rank-sum if any member's `.npz` is missing.

### Speed work

- Two CoAt readers run **in parallel pairs** — `(resgated, d4)` and `(global96, repairv1)`.
- **Raptor header cache** is seeded from the DINOv2 ordering pass, so Raptor never re-reads DICOM headers.

> **Key idea:** beyond picking the best single model per finding, the frontier now comes from combining **more, genuinely different CoAt readers** and **reducing them in probability space** rather than rank space.

---

## 🦖 Work 6 — 0.950
**RSNA Knee V2 — Velciraptor Dinosaur Speed**

🔗 https://www.kaggle.com/code/matterhorn3838/rsna-knee-v2-velciraptor-dinosaur-speed/notebook

### What changed vs 0.943

A major architectural shift: the large multi-family ensemble is replaced by **two complementary MRI readers**, using only two attached weight datasets.

| Component | 0.943 | 0.950 |
|-----------|-------|-------|
| DINOv2 | 20 members | **Removed** |
| A5 | 5 folds | **Removed** |
| RadImageNet | Included | **Removed** |
| CoAtNet | Multi-family ensemble | **One SWA checkpoint** |
| ConvNeXt 2.5D MIL | Not included | **3 checkpoints** |
| DICOM processing | Shared caching | **Optimized fast reader** |
| Final fusion | Multi-stage blending | **Target-specific rank fusion** |
| Attached datasets | 14 dataset versions | **2 datasets** |

### Two complementary readers

- **CoAtNet Raptor** — 96-slice representation at 384 px, 94 overlapping 2.5D windows, SWA weights and anatomical mirror TTA.
- **ConvNeXt 2.5D MIL** — three pretrained checkpoints, six anatomical series slots, adjacent-slice triplets and transformer-based aggregation.
- **Finding-specific attention** — ConvNeXt combines evidence across MRI windows separately for each of the 12 abnormalities.
- **Target-specific fusion** — each finding receives its own CoAtNet/ConvNeXt mixing weight, followed by final percentile ranking.

### Speed and reproducibility

- **Base64-embedded fast DICOM reader** — optimized decoding and preprocessing with the `study_lut` strategy.
- **SHA-256 verification** — validates the CoAtNet checkpoint before inference.
- **Compact pipeline** — 7 code blocks instead of the previous 23.
- **Fallback protection** — preserves the standalone CoAtNet submission if ConvNeXt fusion fails.

> **Key idea:** architectural complementarity can be more valuable than ensemble size. Instead of adding more similar readers, combine two strong representations that may capture different MRI abnormalities — then fuse them separately for each target.

---

## 🔑 Key Pattern

The evolution reveals **two distinct phases**: incremental ensemble refinement followed by architectural simplification.

1. **Diversity** — more CoAtNet arms + reverse-flip.
2. **Inner strengthening** — residual-gated CoAt.
3. **Target-specific routing** — full weight to the best model for a problematic label.
4. **Sampling density** — `k_eval=94` with no new models.
5. **Family depth + probability-space reduction** — 4 CoAt members, reduced in probability space.
6. **Architectural complementarity + simplification** — replace a large ensemble with CoAtNet SWA + ConvNeXt 2.5D MIL and target-specific rank fusion.

### The bigger lesson

From **0.937 to 0.943**, improvements largely came from adding models, increasing sampling density and optimizing ensemble weights.

The jump from **0.943 to 0.950** follows a different direction: **fewer model
---

## 📚 Credits

All models, weights, and pipelines below are public work by other authors. We reproduced them one-to-one and packaged them into a single notebook. Full credit to their authors and contributors.

- **Work 1 (0.937)** — [prvsiyan](https://www.kaggle.com/code/prvsiyan/head-and-shoulders-knees-and-toes)
- **Work 2 (0.94)** — [renta0426](https://www.kaggle.com/code/renta0426/rsna-knee-hybrid-raptor-lateral-meniscus-r100)
- **Work 3 (0.941)** — [Mattia Angeli](https://www.kaggle.com/code/mattiaangeli/bend-the-knee-to-the-dinosaurs) + credits: Pilkwang, Sofia Anjenje, Antoine G., prvsiyan, Marwan Mahmoud, dreaddevelopment/Roman Tamrazov, renta.k, Anvith Pothula
- **Work 4 (0.942)** — [maverick_ss_26](https://www.kaggle.com/code/maverickss26/rsna-knee-0942-restructured/notebook)
- **Work 5 (0.943)** — [Jiwei Liu](https://www.kaggle.com/code/jiweiliu/rsna-knee-fast-2xt4-inference/notebook)
- **Work 6 (0.950)** — [Matterhorn3838](https://www.kaggle.com/code/matterhorn3838/rsna-knee-v2-velciraptor-dinosaur-speed/notebook)

## 1. 🔧 CUDA & CoAtNet Checkpoint Verification

Checks GPU (CUDA) availability and locates the CoAtNet Raptor weights in the attached Kaggle datasets.

**Key components:**
- `RUN_INFERENCE = True` — enables inference execution.
- `_weight_name` — checkpoint file `raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt`.
- `_candidates` — four possible checkpoint locations in Kaggle.
- **SHA-256:** verifies checkpoint integrity against the expected hash `7e5315da...72244ef`.
- Execution stops if CUDA or the checkpoint is unavailable, or if the hash does not match. No alternative weights are substituted automatically.

**Output:** The verified checkpoint path is stored in `_weights` for the next block.

In [ ]:
# Authorized scoring run: verify the selected checkpoint before inference.
RUN_INFERENCE = True
if not RUN_INFERENCE:
    raise SystemExit("Static notebook only. Attach the exact checkpoint and competition, enable a T4 GPU, and set RUN_INFERENCE = True to run inference.")

import hashlib
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("This scored pipeline requires CUDA; CPU inference is not enabled.")
_weight_name = 'raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt'
_candidates = [
    Path('/kaggle/input/rsna-knee-publication-swa-weights-20261007') / _weight_name,
    Path('/kaggle/input/datasets/nartaa/rsna-knee-publication-swa-weights-20261007') / _weight_name,
    Path('/kaggle/input/rsna-knee-oai-original384-s42-pair') / _weight_name,
    Path('/kaggle/input/datasets/nartaa/rsna-knee-oai-original384-s42-pair') / _weight_name,
]
_weights = next((p for p in _candidates if p.is_file()), None)
if _weights is None:
    raise FileNotFoundError("Attach the exact selected weights; no alternative model is substituted.")

with _weights.open('rb') as _f:
    if hasattr(hashlib, "file_digest"):
        digest = hashlib.file_digest(_f, 'sha256').hexdigest()
    else:
        h = hashlib.sha256()
        while chunk := _f.read(65536):
            h.update(chunk)
        digest = h.hexdigest()
    assert digest == '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef', f"Checkpoint hash differs: got {digest}"
print(f"Verified verified SOTA weights: {_weights} (SHA-256 {digest})")

## 2. 🦖 CoAtNet Raptor — MRI Inference Pipeline

Runs the main CoAtNet Raptor inference pipeline using a verified SWA checkpoint and high-resolution MRI volumes.

**Key components:**
- **CoAtNet RMLP-2** — pretrained backbone with an SWA checkpoint.
- `IMG = 384` — native 384 × 384 MRI resolution.
- `MAXS = 96` — standardized 96-slice representation across anatomical series.
- `K_EVAL = 94` — overlapping 2.5D windows covering the study.
- **Mirror TTA** — anatomical left/right augmentation for more robust predictions.
- `_fastread.py` — Base64-embedded accelerated DICOM reader with parallel preprocessing.
- `rankpct()` — converts predictions into percentile ranks for all 12 targets.

**Output:** Saves `submission.csv` and a standalone CoAtNet backup `_sota_0949.csv` for the final fusion stage.

In [ ]:
# %% [code]
#!/usr/bin/env python3
"""Knee MRI: twelve findings from a single model

This notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,
medial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral
compartments, joint effusion, synovitis, a Baker's cyst, bone contusion and fracture. It scores
0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.

This is the inference half of the work. The model was trained separately and its weights are
attached as a dataset, so this notebook only loads them and predicts:
https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense

Where the training labels came from

Worth saying up front, because it shapes everything else. The competition gives you 4,407 studies
but structured labels for only 58 of them. Every other study arrives with a free-text radiology
report and nothing more, so there is very little to train against out of the box.

The labels behind these weights were made by reading those reports with a language model and
turning each into twelve probabilities rather than twelve yes or no answers. A report that says a
tear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every
hedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came
with real labels were never trained on and are used to check the result honestly; the model reaches
0.9167 macro-AUC on them.

Building a fixed input from studies that are all shaped differently

The hard part of this competition is not the network, it is that no two studies look alike. A
study holds several DICOM series shot in different planes, the number of series varies, and the
number of slices in a series varies more. Anything that expects a fixed-size input has to be given
one.

The approach here is to fill five fixed slots per study, always in the same order, for a stack of
64 images:

  18 slices from a sagittal series, preferring a fluid-sensitive one
  14 slices from a second sagittal series, preferring one that is not fluid-sensitive
  12 slices from a coronal series, preferring a fluid-sensitive one
   8 slices from a second coronal series
  12 slices from an axial series

Preferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-
sensitive sequences show swelling, effusion and acute injury clearly, while the other sequences
show anatomy and cartilage better, and the twelve findings are split across both. If a study has
no series for a slot, the slot is left as zeros and the model is told to skip it rather than being
fed something misleading.

Within a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the
middle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and
cutting them was measurably costing accuracy on exactly those findings.

Every slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from
the DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count
matters: it means a knee occupies the same fraction of the frame whether the scan was acquired at
0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical
information.

How the model reads the stack

Three neighbouring slices are stacked into the three channels of one image. The network then sees
a little of what lies above and below the slice in the middle, which is most of the benefit of a 3D
model at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet
backbone at 384 pixels.

The windows are combined with an attention layer that has separate weights for each of the twelve
findings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices
while osteoarthritis is spread across many coronal ones, and a single pooled score forces those two
to share one notion of which slices are important. Giving each finding its own attention weights
lets each one draw on the slices that actually show it.

Running it

Scoring uses 42 windows per study. Inference runs in half precision and automatically retries a
study in full precision if it fails, so no study is ever dropped from the submission. The notebook
needs no internet: the backbone is loaded from the attached weights rather than downloaded.
"""
import os, sys, glob, time, json, gc
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import threading, queue
import torch, torch.nn as nn, torch.nn.functional as F
import timm
# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes
# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for
# the fixed (1,24,3,res,res) input.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- fixed config (must match training exactly) -----------------------------
INPUT_RES = 320
INPUT_MODE = "crop"
IMG = 384   # native 384 corpus: the model saw 384 px crops directly, no 336->384 upsample
CROP_MM = 140.0
# 96 slices per study at native 384 px. Must match the corpus the weights were trained on
# (cache/build_corpus96.py).
SLOTS = [("Sagittal", 1, 26), ("Sagittal", 0, 22), ("Coronal", 1, 18),
         ("Coronal", 0, 12), ("Axial", -1, 18)]
MAXS = sum(s[2] for s in SLOTS)                     # 96
K_EVAL = 94   # every window position of the 96-slice test volume (full coverage)
HDR_THREADS = 16   # per prep process: header reads are fuse-latency-bound, not CPU-bound
PX_THREADS = 8     # per prep process: pixel reads of the ~96 picked slices
QUEUE_MAX = 48     # studies in flight between the prep pool and the GPU consumers (RAM bound: ~14 MB each)
NORM = "imagenet"
FASTREAD_VARIANT = 'study_lut'   # kernel v3: research/fastread/readers.py, byte-identical to build_study
FR_HDR, FR_PX = 32, 16
_FASTREAD_B64 = "..."
def _load_fastread():
    import base64 as _b, importlib.util as _iu
    p = '/kaggle/working/_fastread.py' if os.path.isdir('/kaggle/working') else os.path.join(os.path.dirname(os.path.abspath(__file__)), '_fastread.py')
    if not os.path.exists(p):
        open(p, 'wb').write(_b.b64decode(_FASTREAD_B64))
    spec = _iu.spec_from_file_location('_fastread', p); m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
    return m
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.
# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm
# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.
# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.
# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).
# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),
# cnlarge384 (0.8752), maxvit384 (0.8438).
#
# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.
#
# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while
# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed
# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.
#
# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45
# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).
#
# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):
#   incumbent CoAtNet (3,155-study corpus) 0.8923
#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)
# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,
# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.
ARMS = [   # our 5-fold CoAtNet (public recipe reproduced on our folds/labels), equal-weight rank-mean
    {"file": "raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt", "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384, "w": 1.0},
]


# ============================================================================
# Model -- verbatim from finetune_raptor.py
# ============================================================================
def build_backbone(arch, pretrained=False):
    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable
    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them
    # down the ViT path (mirrors finetune_raptor.py exactly).
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3, img_size=INPUT_RES)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        # Bound backbone memory; the attention head still sees every window together.
        flat = x.flatten(0, 1)
        f = torch.cat([self.backbone(chunk) for chunk in flat.split(16)], dim=0)
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))


def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    import hashlib
    with open(pt_path, "rb") as stream:
        observed_sha = hashlib.file_digest(stream, "sha256").hexdigest()
    assert observed_sha == '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef', "Bound checkpoint bytes differ"
    print(f"[weights] sha256 {observed_sha}", flush=True)
    ck = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch = ck.get("arch", arch_default)
    assert int(ck.get("res", res_default)) == 384, "Expected original 384 checkpoint"
    ck_res = INPUT_RES
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM
    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows
    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.
    del ck
    gc.collect()
    return model, ck_res


# ============================================================================
# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)
# ============================================================================
def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]


SLOT_SIZES = [26, 22, 18, 12, 18]
SAGITTAL_SLOTS = (0, 1)

def _slot_index(D):
    b = np.cumsum([0] + SLOT_SIZES)
    if b[-1] != D: raise RuntimeError(f'slot sizes sum to {b[-1]}, volume has {D} slices')
    s_ = np.searchsorted(b, np.arange(D), side='right') - 1
    rel = (np.arange(D) - b[s_]) / np.maximum(1, np.array(SLOT_SIZES)[s_] - 1)
    return s_, rel

def _mirror_tri(tri, cs, D):
    """final-g1-bundle MIRROR_TTA: true L/R mirror of (k,3,H,W) uint8 eval triplets (research/rnd4/mirror/mirror_f0.py). The slot of each
    window's CENTRE slice decides the axis: sagittal (slots 0,1) -> reverse the 3 slice-channels; coronal/axial (2,3,4) -> flip the width.
    Done on uint8 BEFORE the per-channel LUT, i.e. 'mirror the knee, then normalise'."""
    s_, _ = _slot_index(D)
    sag = torch.from_numpy(np.isin(s_[np.asarray(cs, dtype=np.int64)], SAGITTAL_SLOTS)).to(tri.device).view(-1, 1, 1, 1)
    return torch.where(sag, tri.flip(1), tri.flip(-1))

def infer_pair_probs(model, plain, mirrored, device):
    assert plain.shape==mirrored.shape==(94,3,320,320)
    assert all(not m.training for m in model.modules()), "TTA must not update BN or enable dropout"
    p = infer_probs(model, plain, device)
    q = infer_probs(model, mirrored, device)
    return 0.5 * (p + q)

def eval_windows(vol, mask, k, res, norm=NORM, mirror=False):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0)
        if mirror:
            tri = _mirror_tri(torch.from_numpy(tri)[None], [c], D)[0].numpy()
        tri = tri.astype(np.float32) / 255.0
        assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES
        lo = (IMG - res) // 2
        tri = tri[:, lo:lo + res, lo:lo + res]
        t = torch.from_numpy(np.ascontiguousarray(tri))
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear",
                              align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == "imagenet":
        x = (x - _MEAN) / _STD
    return x


_WLUT = {}
def gpu_windows(vol, mask, k, res, dev, norm=NORM, mirror=False):
    """Same windows as eval_windows, built on the GPU from the uint8 volume. Values come from a per-channel
    256-entry table computed on the CPU with eval_windows' own float32 ops, so they are bit-identical."""
    D = vol.shape[0]
    cs = np.array([max(1, min(c, D - 2)) for c in _eval_centers(mask, D, k)])
    tri = torch.from_numpy(np.ascontiguousarray(vol[np.stack([cs - 1, cs, cs + 1], 1)])).to(dev)   # (k,3,IMG,IMG) uint8
    if mirror:
        tri = _mirror_tri(tri, cs, D)
    assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES
    lo = (IMG - res) // 2
    tri = tri[:, :, lo:lo + res, lo:lo + res]
    key = (str(dev), norm)
    if key not in _WLUT:
        v = torch.from_numpy((np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0))
        t = torch.stack([v, v, v], 0).view(3, 256, 1)
        if norm == 'imagenet':
            t = (t - _MEAN.view(3, 1, 1)) / _STD.view(3, 1, 1)
        _WLUT[key] = t.view(3, 256).contiguous().to(dev)
    L = _WLUT[key]
    x = torch.stack([L[c][tri[:, c].long()] for c in range(3)], 1)                   # exact float32 values
    if x.shape[-1] != res:
        if norm == 'imagenet':   # resize must happen BEFORE normalisation, as in eval_windows
            L0 = _WLUT.setdefault((str(dev), 'none'), torch.from_numpy(np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0).to(dev))
            x = L0[tri.long()]
            x = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
            x = (x - _MEAN.to(dev)) / _STD.to(dev)
        else:
            x = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
    return x


@torch.no_grad()
def infer_probs(model, xwins, device):
    assert str(device).startswith("cuda"), "Qualified inference requires CUDA/fp16"
    x = xwins.unsqueeze(0).to(device)
    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(x).float()
        assert torch.isfinite(logits).all(), "Nonfinite fp16 logits"
        o = torch.sigmoid(logits)
    return o[0].cpu().numpy()


def rankpct(x):                                   # per-column percentile rank in [0,1]
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))


# ============================================================================
# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def _hdr(f):
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True)
            iop = getattr(h, 'ImageOrientationPatient', None)
            ipp = getattr(h, 'ImagePositionPatient', None)
            if iop is not None and ipp is not None and len(iop) == 6:
                r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
            else:
                pos = float(getattr(h, 'InstanceNumber', 0) or 0)
            ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
            return (pos, f, ps, True)
        except Exception:
            return (0.0, f, 0.5, False)

    def order_and_meta(sdir):
        # Header reads are latency-bound on Kaggle's mounted competition data (~240k opens on the hidden
        # set); a thread pool hides that latency. Records and sort are unchanged -> identical ordering.
        fs = glob.glob(sdir + "/*.dcm")
        with ThreadPoolExecutor(max_workers=HDR_THREADS) as ex:
            out = list(ex.map(_hdr, fs))
        recs = [(pos, f, ps) for pos, f, ps, _ in out]; ps_list = [ps for _, _, ps, ok in out if ok]
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    return order_and_meta, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def build_study(sid, ser_records, tsdir, reader):
    order_and_meta, read_px, mm_crop_resize = reader
    rows = ser_records.get(sid, [])
    vol = np.zeros((MAXS, IMG, IMG), np.uint8); idx = 0; used = set()
    for plane, fluid, k in SLOTS:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k; continue
        used.add(r['SeriesInstanceUID'])
        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r['SeriesInstanceUID']}")
        if not files:
            idx += k; continue
        # wide span: the collateral ligaments and lateral meniscus live in the
        # peripheral slices the old 0.15-0.85 crop threw away. Must match the corpus
        # the weights were trained on (build_corpus80.py, SPAN).
        n = len(files); lo, hi = int(n * 0.02), int(n * 0.98) - 1; hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        sel = [files[min(p, n - 1)] for p in picks]
        def _px(fp_ps):
            fp, ps = fp_ps
            try: return read_px(fp), ps
            except Exception: return None, med_ps
        with ThreadPoolExecutor(max_workers=PX_THREADS) as ex:
            got = list(ex.map(_px, sel))                    # order preserved
        arrs = [a for a, _ in got]; pss = [ps for _, ps in got]
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [2.0, 98.0])
        else:
            loq, hiq = 0.0, 1.0
        for a, ps in zip(arrs, pss):
            if idx >= MAXS: break
            if a is None: idx += 1; continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)
            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)
            vol[idx] = (aw * 255).astype(np.uint8); idx += 1
        if idx >= MAXS: break
    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask



# ---- parallel preprocessing helpers (module level so they pickle under fork) ----
_SER = None; _TSDIR = None; _READER = None
def _pre_init():
    global _READER
    _READER = _load_fastread().make_builder(FASTREAD_VARIANT, IMG, SLOTS, hdr_threads=FR_HDR, px_threads=FR_PX)
def _pre_one(sid):
    try:
        vol, mask = _READER(sid, _SER, _TSDIR)
    except Exception as e:
        print(f"  [pre] {sid[:16]} FAILED ({type(e).__name__}: {e})", flush=True)
        vol = np.zeros((MAXS, IMG, IMG), np.uint8); mask = np.zeros(MAXS, np.uint8)
    return sid, vol, mask

# ============================================================================
# Test-root discovery + weights + main
# ============================================================================
def find_test_root():
    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
             "/kaggle/input/rsna-knee-abnormality-detection"]
    for b in cands:
        if os.path.exists(b + "/test.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):
            return d
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f:
            return d
    raise RuntimeError("no test root under /kaggle/input")


def find_weight_file(fname):
    # direct dataset mounts first; NEVER recursive-glob the competitions DICOM tree.
    direct = [f"/kaggle/input/raptor-knee-arms/{fname}",
              f"/kaggle/input/raptor-knee-arms/1/{fname}",
              f"/kaggle/input/raptor-cnn336/{fname}"]
    for p in direct:
        if os.path.exists(p):
            return p
    for d in sorted(glob.glob("/kaggle/input/*/")):
        # skip the competition data wherever Kaggle mounts it (/kaggle/input/competitions/<slug> OR /kaggle/input/<slug>):
        # recursively globbing its DICOM tree hung a commit run for 20+ min (2026-09-26)
        if "competition" in d.lower() or "abnormality-detection" in d or any(
                os.path.exists(os.path.join(d, x)) for x in ("test_series", "train_images", "train.csv", "test.csv")):
            continue
        hits = glob.glob(os.path.join(d, "**", fname), recursive=True)
        if hits:
            return hits[0]
    raise RuntimeError(f"{fname} not found under /kaggle/input")


def main():
    import pandas as pd
    t0 = time.time()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ngpu = torch.cuda.device_count()
    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)

    ROOT = find_test_root()
    tsdir = ROOT + "/test_series"
    if not os.path.isdir(tsdir):
        tsdir = ROOT + "/test_images"
    print("test root:", ROOT, "| series dir:", tsdir, flush=True)

    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    test_ids = test["StudyInstanceUID"].tolist()
    tser = pd.read_csv(ROOT + "/test_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)

    sub_cols = ["StudyInstanceUID"] + LAB
    ssub = os.path.join(ROOT, "sample_submission.csv")
    if os.path.exists(ssub):
        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)

    N = len(test_ids); A = len(ARMS)
    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]
    pos = {sid: i for i, sid in enumerate(test_ids)}

    # ---- 2. GPU consumers: one thread per GPU, every arm loaded on every GPU (per-study batch of 1,
    #         so the math is identical to the sequential kernel; only the wall-clock changes) ----
    xla = None
    try:
        import torch_xla; import torch_xla.core.xla_model as xm
        dev = torch_xla.device(); xla = xm
        print("device: XLA/TPU", dev, flush=True)
    except Exception as e:
        print("XLA unavailable:", type(e).__name__, str(e)[:80], "| device", dev, flush=True)
    devs = [f"cuda:{g}" for g in range(ngpu)] if (xla is None and ngpu > 0) else [dev]
    models = {}
    for d in devs:
        models[d] = []
        for a, arm in enumerate(ARMS):
            model, res = load_model(find_weight_file(arm["file"]), arm["arch"], arm["res"], d)
            models[d].append((model, res))
        print(f"[gpu {d}] loaded {A} arm(s) | {time.time()-t0:.0f}s", flush=True)
    q = queue.Queue(maxsize=QUEUE_MAX); done_n = [0]; lock = threading.Lock()

    def consumer(d):
        while True:
            item = q.get()
            if item is None: q.task_done(); return
            sid, vol, mask = item; i = pos[sid]
            try:
                for a, (model, res) in enumerate(models[d]):
                    xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM) if xla is not None else gpu_windows(vol, mask, K_EVAL, res, d)
                    assert xla is None, "Anatomical L/R TTA requires CUDA/fp16"
                    xm = gpu_windows(vol, mask, K_EVAL, res, d, norm=NORM, mirror=True)
                    arm_probs[a][i] = infer_pair_probs(model, xw, xm, d)
                    del xm
            except Exception as e:
                print(f"  [gpu {d}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})", flush=True)
            with lock:
                done_n[0] += 1
                if done_n[0] % 200 == 0 or done_n[0] == N:
                    print(f"  [gpu] {done_n[0]}/{N} | {time.time()-t0:.0f}s", flush=True)
            q.task_done()

    threads = [threading.Thread(target=consumer, args=(d,), daemon=True) for d in devs]
    for th in threads: th.start()

    # ---- 3. prep every study ONCE in a process pool and stream it to the GPUs as it lands ----
    import multiprocessing as mp
    global _SER, _TSDIR, _READER
    _SER, _TSDIR = SER, tsdir
    nproc = max(4, min(96, (os.cpu_count() or 8) - 8))
    t_pre = time.time()
    with mp.get_context("fork").Pool(nproc, initializer=_pre_init) as pool:
        for i, (sid, vol, mask) in enumerate(pool.imap_unordered(_pre_one, test_ids, chunksize=2)):
            q.put((sid, vol, mask))
            if (i + 1) % 200 == 0 or i + 1 == N:
                print(f"  [pre] {i+1}/{N} | {time.time()-t0:.0f}s", flush=True)
    print(f"[pre] done: {N} studies in {time.time()-t_pre:.0f}s (procs {nproc} x fastread {FASTREAD_VARIANT} {FR_HDR}/{FR_PX} threads)", flush=True)
    for _ in devs: q.put(None)
    q.join()
    for th in threads: th.join()
    print(f"[gpu] done: {done_n[0]}/{N} studies on {len(devs)} device(s) | {time.time()-t0:.0f}s", flush=True)
    for d in devs:
        for m, _ in models[d]: del m
    models.clear(); gc.collect()

    # WEIGHTED rank-mean blend across the test set, per finding (the offline recipe).
    # Weights come from ARMS[*]["w"] and are normalised here, so dropping/adding an arm can
    # never silently change the scale. Falls back to equal weights if none are declared.
    _w = np.array([float(a.get("w", 1.0)) for a in ARMS], dtype=np.float64)
    _w = _w / _w.sum()
    print(f"[blend] weighted rank-mean w={dict(zip([a['file'] for a in ARMS], _w.round(4)))}", flush=True)
    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]),
                         axes=(0, 0))                                          # (N,12) in [0,1]
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5

    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)
    sub.insert(0, "StudyInstanceUID", test_ids)
    sub = sub[sub_cols]
    assert list(sub.columns) == sub_cols, "column order drift"
    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
    assert np.isfinite(sub[LAB].values).all()
    out = "/kaggle/working/submission.csv"
    sub.to_csv(out, index=False)
    sub.to_csv("/kaggle/working/_sota_0949.csv", index=False)
    print("Saved SOTA 0.949 backup to /kaggle/working/_sota_0949.csv", flush=True)
    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)
    print(sub.head().to_string(index=False), flush=True)
    print(f"DONE {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()

## 3. 📁 ConvNeXt Reader — Directory Setup

Creates a separate source directory for the complementary ConvNeXt 2.5D MIL inference pipeline.

**Key components:**
- `os.makedirs()` — creates the directory if it does not already exist.
- `/kaggle/working/own_src/` — stores the Python modules used by the second reader.
- The directory is shared by the preprocessing, model architecture, and inference modules in Blocks 4–6.

**Output:** Creates `/kaggle/working/own_src/` without modifying existing files.

In [ ]:
os.makedirs('/kaggle/working/own_src', exist_ok=True)
print('Created /kaggle/working/own_src directory.')

## 4. 🧩 DICOM Preprocessing & MRI Standardization

Defines the ConvNeXt preprocessing module for converting raw DICOM series into standardized MRI volumes.

**Key components:**
- `preprocess.py` — standalone DICOM loading and normalization module.
- `MM_PER_PX = 0.4` — standardized physical pixel spacing.
- `SIZE = 384` — 384 × 384 images with a 153.6 mm field of view.
- `MAX_SLICES = 64` — maximum number of slices per series.
- `CANON` — anatomical orientation and slice-order standardization for sagittal, coronal, and axial planes.
- **Intensity normalization** — DICOM rescale handling, MONOCHROME1 inversion, and per-series intensity windowing.
- `load_series()` — produces standardized uint8 MRI volumes.

**Output:** Writes `/kaggle/working/own_src/preprocess.py`, which is imported by the ConvNeXt inference pipeline.

In [ ]:
%%writefile /kaggle/working/own_src/preprocess.py
"""Turn each DICOM series into a canonical uint8 volume [n_slices, SIZE, SIZE].

Canonical means: fixed physical pixel size, centre crop to a fixed field of view,
standard in-plane orientation, standard slice order and a per-series intensity window.
The same `load_series` is used for the training cache and for inference.

Usage: python src/preprocess.py [train|test] [n_series_limit]
"""
import os
import sys
from multiprocessing import Pool

import cv2
import numpy as np
import pandas as pd
import pydicom

DATA = "/mnt/ssd/rsna_knee/data"
WORK = "/mnt/ssd/rsna_knee/work"

MM_PER_PX = 0.4
SIZE = 384  # 153.6 mm field of view
MAX_SLICES = 64

# plane -> (normal axis, slice-order sign, (col axis, sign), (row axis, sign)) in LPS coordinates
# Sagittal: slices right->left, image columns anterior->posterior, rows superior->inferior
# Coronal:  slices anterior->posterior, columns patient-right->left, rows superior->inferior
# Axial:    slices superior->inferior, columns patient-right->left, rows anterior->posterior
CANON = {
    0: (+1, (1, +1), (2, -1)),
    1: (+1, (0, +1), (2, -1)),
    2: (-1, (0, +1), (1, +1)),
}
PLANES = ["Sagittal", "Coronal", "Axial"]


def _read(path):
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array.astype(np.float32)
    if ds.get("PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    slope, inter = ds.get("RescaleSlope"), ds.get("RescaleIntercept")
    if slope is not None and inter is not None:
        arr = arr * float(slope) + float(inter)
    iop = [float(v) for v in ds.ImageOrientationPatient] if "ImageOrientationPatient" in ds else None
    ipp = [float(v) for v in ds.ImagePositionPatient] if "ImagePositionPatient" in ds else None
    ps = [float(v) for v in ds.PixelSpacing] if "PixelSpacing" in ds else None
    inst = float(ds.get("InstanceNumber") or 0)
    frames = list(arr) if arr.ndim == 3 else [arr]
    return [(f, iop, ipp, ps, inst + i * 1e-3) for i, f in enumerate(frames)]


def _canon_slice(img, iop, ps, plane):
    """Reorient one slice to the canonical in-plane orientation; returns (img, (row_mm, col_mm))."""
    ps = ps or [MM_PER_PX * img.shape[0] / SIZE] * 2  # no spacing: assume the image spans the crop
    if iop is None:
        return img, ps
    r, c = np.array(iop[:3]), np.array(iop[3:])
    _, (cax, csgn), (rax, rsgn) = CANON[plane]
    if abs(r[cax]) < abs(r[rax]):  # rows and columns are swapped relative to canonical
        img, r, c, ps = img.T, c, r, ps[::-1]
    if r[cax] * csgn < 0:
        img = img[:, ::-1]
    if c[rax] * rsgn < 0:
        img = img[::-1]
    return img, ps


def _resample(img, ps):
    """Centre crop to SIZE*MM_PER_PX mm, resample to MM_PER_PX, zero-pad to SIZE x SIZE."""
    out = np.zeros((SIZE, SIZE), np.float32)
    h, w = img.shape
    ch = min(h, int(round(SIZE * MM_PER_PX / ps[0])))
    cw = min(w, int(round(SIZE * MM_PER_PX / ps[1])))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])
    nh = max(1, min(SIZE, int(round(ch * ps[0] / MM_PER_PX))))
    nw = max(1, min(SIZE, int(round(cw * ps[1] / MM_PER_PX))))
    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR
    res = cv2.resize(crop, (nw, nh), interpolation=interp)
    oy, ox = (SIZE - nh) // 2, (SIZE - nw) // 2
    out[oy:oy + nh, ox:ox + nw] = res
    return out


def load_series(sdir, plane_name=None):
    """Returns (uint8 volume [n, SIZE, SIZE], info dict). Unreadable slices are skipped."""
    slices, bad = [], 0
    for fn in os.listdir(sdir):
        try:
            slices.extend(_read(os.path.join(sdir, fn)))
        except Exception:
            bad += 1
    info = {"n_raw": len(slices), "n_bad": bad}
    if not slices:
        return np.zeros((1, SIZE, SIZE), np.uint8), info

    iop = next((s[1] for s in slices if s[1] is not None), None)
    if iop is not None:
        normal = np.cross(iop[:3], iop[3:])
        plane = int(np.abs(normal).argmax())
    else:
        normal, plane = None, PLANES.index(plane_name) if plane_name in PLANES else 0
    info["plane"] = PLANES[plane]

    if normal is not None and all(s[2] is not None for s in slices):
        sgn = CANON[plane][0] * np.sign(normal[plane])
        slices.sort(key=lambda s: float(np.dot(s[2], normal)) * sgn)
        pos = [float(np.dot(s[2], normal)) for s in slices]
        info["slice_gap"] = float(np.median(np.abs(np.diff(pos)))) if len(pos) > 1 else 0.0
    else:
        slices.sort(key=lambda s: s[4])
    if len(slices) > MAX_SLICES:
        keep = np.linspace(0, len(slices) - 1, MAX_SLICES).round().astype(int)
        slices = [slices[i] for i in keep]

    vol = np.stack([_resample(*_canon_slice(s[0], s[1], s[3], plane)) for s in slices])
    lo, hi = np.percentile(vol[:, ::4, ::4], [1, 99.5])
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)
    info["n"] = len(vol)
    return (vol * 255).round().astype(np.uint8), info


def _job(args):
    split, study, series, plane, out_dir = args
    out = f"{out_dir}/{series}.npy"
    info = {"study": study, "series": series}
    if os.path.exists(out):
        return None
    try:
        vol, extra = load_series(f"{DATA}/{split}_series/{study}/{series}", plane)
        np.save(out + ".tmp.npy", vol)
        os.replace(out + ".tmp.npy", out)
        info.update(extra)
    except Exception as e:
        info["error"] = f"{type(e).__name__}: {e}"[:200]
    return info


def main():
    split = sys.argv[1] if len(sys.argv) > 1 else "train"
    limit = int(sys.argv[2]) if len(sys.argv) > 2 else None
    out_dir = f"{WORK}/cache{SIZE}_{split}"
    os.makedirs(out_dir, exist_ok=True)
    se = pd.read_csv(f"{DATA}/{split}_series.csv")
    if limit:
        se = se.sample(limit, random_state=0)
    tasks = [(split, a, b, p, out_dir) for a, b, p in zip(se.StudyInstanceUID, se.SeriesInstanceUID, se.Anatomical_Plane)]
    rows = []
    with Pool(18) as p:
        for i, r in enumerate(p.imap_unordered(_job, tasks, chunksize=4)):
            if r:
                rows.append(r)
            if i % 1000 == 0:
                print(f"{i}/{len(tasks)}", flush=True)
    df = pd.DataFrame(rows)
    meta = f"{WORK}/cache{SIZE}_{split}_meta.csv"
    if os.path.exists(meta) and len(df):
        df = pd.concat([pd.read_csv(meta), df]).drop_duplicates("series", keep="last")
    if len(df):
        df.to_csv(meta, index=False)
        print("done", len(df), "errors:", int(df["error"].notna().sum()) if "error" in df else 0)


if __name__ == "__main__":
    main()

## 5. 🧠 ConvNeXt 2.5D MIL — Model Architecture

Defines the ConvNeXt-based multiple-instance learning model for predicting all 12 knee abnormalities.

**Key components:**
- `knee.py` — shared model architecture and dataset utilities.
- `N_SLOTS = 6` — three anatomical planes × two fat-suppression categories.
- `build_study_table()` — groups MRI series by study and anatomical slot, with fallback for missing fat-suppression variants.
- `window_centres()` — selects slice positions for 2.5D input windows.
- **ConvNeXt backbone** — extracts features from three adjacent MRI slices.
- **Transformer aggregation** — combines information across slices and anatomical series.
- **Finding-specific attention** — produces separate predictions for all 12 targets.

**Output:** Writes `/kaggle/working/own_src/knee.py` containing the `KneeNet` architecture and supporting utilities.

In [ ]:
%%writefile /kaggle/working/own_src/knee.py
"""Shared dataset and model code for training and inference."""
import math

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
PLANES = ["Sagittal", "Coronal", "Axial"]
# slot = plane index + 3 * (1 - fat_suppressed): fat-suppressed planes first
N_SLOTS = 6
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def slot_of(plane, fat_sup):
    return PLANES.index(plane) + 3 * (1 - int(fat_sup))


def build_study_table(series_df, n_slices):
    """series_df: StudyInstanceUID, SeriesInstanceUID, Fat_Suppression, Anatomical_Plane.
    n_slices: dict series -> cached slice count. Returns {study: [[series ids of slot 0], ...]}
    with each slot's candidates sorted by slice count (largest first)."""
    table = {}
    for st, sr, fs, pl in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID,
                              series_df.Fat_Suppression, series_df.Anatomical_Plane):
        if n_slices.get(sr, 0) < 3:
            continue
        table.setdefault(st, [[] for _ in range(N_SLOTS)])[slot_of(pl, fs)].append(sr)
    for slots in table.values():
        for cands in slots:
            cands.sort(key=lambda s: -n_slices[s])
        # Intelligent anatomical plane fallback: if fat-sat is missing, use non-fat-sat (and vice-versa)
        for p in range(3):
            fs_s, nonfs_s = p, p + 3
            if not slots[fs_s] and slots[nonfs_s]:
                slots[fs_s] = list(slots[nonfs_s])
            elif not slots[nonfs_s] and slots[fs_s]:
                slots[nonfs_s] = list(slots[fs_s])
    return table


def window_centres(n, k, train, lo=0.04, hi=0.96):
    """k slice indices (centres of 3-slice windows) spread over the series."""
    a, b = max(1.0, lo * (n - 1)), min(n - 2.0, hi * (n - 1))
    if b < a:
        a = b = (n - 1) / 2
    if train:
        edges = np.linspace(a, b, k + 1)
        c = edges[:-1] + np.random.rand(k) * (edges[1:] - edges[:-1])
    else:
        c = np.linspace(a, b, k)
    return np.clip(np.round(c).astype(int), 1, max(1, n - 2))


class StudyDataset(torch.utils.data.Dataset):
    """One item = one study: uint8 windows [N_SLOTS, k, 3, H, W], slot mask, slice positions, target."""

    def __init__(self, studies, table, cache_dir, targets=None, k=8, train=False, loader=None):
        self.studies, self.table, self.cache_dir = list(studies), table, cache_dir
        self.targets, self.k, self.train = targets, k, train
        self.loader = loader or (lambda s: np.load(f"{self.cache_dir}/{s}.npy", mmap_mode="r"))

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, i):
        st = self.studies[i]
        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])
        x, mask, pos = None, np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)
        for s, cands in enumerate(slots):
            if not cands:
                continue
            sr = cands[np.random.randint(len(cands))] if self.train else cands[0]
            vol = self.loader(sr)
            n = len(vol)
            c = window_centres(n, self.k, self.train)
            win = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])  # [k, 3, H, W]
            if x is None:
                x = np.zeros((N_SLOTS, self.k) + win.shape[1:], np.uint8)
            x[s], mask[s], pos[s] = win, True, c / max(n - 1, 1)
        if x is None:
            x = np.zeros((N_SLOTS, self.k, 3, 384, 384), np.uint8)
        y = self.targets[i] if self.targets is not None else np.zeros(len(LABELS), np.float32)
        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos), torch.from_numpy(np.asarray(y, np.float32))


def make_views(x, res, train, group, fov=0.92):
    """x: uint8 [N, 3, H, W] with N = n_series * group. Crops a centred `fov` fraction of the cached
    field of view and resizes to res; in training adds rotation/scale/shift and gain/gamma jitter
    that is shared within a series."""
    n = x.shape[0]
    ns = n // group
    dev = x.device
    x = x.float().div_(255.0)
    if train:
        ang = (torch.rand(ns, device=dev) - 0.5) * 2 * math.radians(12)
        sc = fov * (1 + (torch.rand(ns, device=dev) - 0.5) * 0.24)
        tx = (torch.rand(ns, device=dev) - 0.5) * 0.12
        ty = (torch.rand(ns, device=dev) - 0.5) * 0.12
        gamma = torch.exp((torch.rand(ns, device=dev) - 0.5) * 0.5)
        gain = 1 + (torch.rand(ns, device=dev) - 0.5) * 0.2
    else:
        ang = torch.zeros(ns, device=dev)
        sc = torch.full((ns,), fov, device=dev)
        tx = ty = torch.zeros(ns, device=dev)
        gamma = gain = None
    cos, sin = torch.cos(ang) * sc, torch.sin(ang) * sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    theta = theta.repeat_interleave(group, 0)
    grid = F.affine_grid(theta, (n, 3, res, res), align_corners=False)
    if x.shape[-1] > 1.5 * res:  # avoid aliasing when shrinking a lot
        x = F.avg_pool2d(x, 2) if x.shape[-1] >= 2 * res else F.interpolate(x, scale_factor=0.75, mode="bilinear", antialias=True)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    if train:
        g = gamma.repeat_interleave(group).view(n, 1, 1, 1)
        x = x.clamp_min(1e-4).pow(g) * gain.repeat_interleave(group).view(n, 1, 1, 1)
    mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)
    return (x - mean) / std


class KneeNet(nn.Module):
    """2D backbone on 3-slice windows -> tokens tagged with series slot and slice position ->
    small transformer over all windows of the study -> one attention pooling per finding."""

    def __init__(self, backbone="convnext_tiny", pretrained=True, dim=384, depth=2, drop=0.2, drop_path=0.1,
                 img_size=None):
        super().__init__()
        kw = {"drop_path_rate": drop_path} if drop_path else {}
        if img_size and backbone.startswith(("vit_", "eva", "beit")):  # fixed-grid transformers need the input size
            kw["img_size"] = img_size
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, **kw)
        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(self.enc.num_features, dim), nn.LayerNorm(dim))
        self.slot_emb = nn.Embedding(N_SLOTS, dim)
        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))
        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None
        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, len(LABELS)))
        self.cls_w = nn.Parameter(torch.randn(len(LABELS), dim) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(len(LABELS)))

    def head(self, feat, mask, pos):
        """feat [B, S, K, D], mask [B, S] bool, pos [B, S, K] in 0..1 -> logits [B, 12]."""
        b, s, k, _ = feat.shape
        slot = torch.arange(s, device=feat.device).view(1, s, 1).expand(b, s, k)
        h = self.proj(feat) + self.slot_emb(slot) + self.pos_emb(pos.unsqueeze(-1).to(feat.dtype))
        h = h.view(b, s * k, -1)
        pad = ~mask.unsqueeze(-1).expand(b, s, k).reshape(b, s * k)
        pad = pad & ~pad.all(1, keepdim=True)  # a study with no series at all attends to everything
        if self.tf is not None:
            h = self.tf(h, src_key_padding_mask=pad)
        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)  # [B, T, C]
        pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)
        return (pooled * self.cls_w).sum(-1) + self.cls_b

    def forward(self, x, mask, pos, res=256, train=False):
        """x uint8 [B, S, K, 3, H, W]."""
        b, s, k = x.shape[:3]
        keep = mask.view(-1)  # run the backbone only on series that exist
        xv = x.view(b * s, k, *x.shape[3:])[keep].flatten(0, 1)
        f = self.enc(make_views(xv, res, train, k).contiguous(memory_format=torch.channels_last))
        feat = f.new_zeros(b * s, k, f.shape[-1])
        feat[keep] = f.view(-1, k, f.shape[-1])
        return self.head(feat.view(b, s, k, -1), mask, pos)

## 6. 🔬 ConvNeXt Reader — DICOM Inference Engine

Defines the inference engine for generating study-level predictions with the pretrained ConvNeXt 2.5D MIL checkpoints.

**Key components:**
- `infer.py` — standalone inference module for the ConvNeXt reader.
- `DicomStudyDataset` — loads selected MRI series and builds 2.5D input windows.
- `k = 16` — 16 windows per anatomical slot, using three adjacent slices each.
- `load_models()` — loads the supplied ConvNeXt checkpoints.
- **CUDA FP16 autocast** — accelerates GPU inference.
- **Checkpoint ensemble** — averages sigmoid probabilities across loaded models.
- `StudyInstanceUID` — preserves study identifiers for correct prediction alignment.

**Output:** Writes `/kaggle/working/own_src/infer.py`. When executed by Block 7, this module generates ConvNeXt predictions for all 12 targets.

In [ ]:
%%writefile /kaggle/working/own_src/infer.py
"""Predict the 12 findings for every study straight from DICOM folders.

Used by the Kaggle submission notebook and locally to check that predictions from raw DICOMs
match those from the training cache.

Usage: python src/infer.py <data_dir> <split> <out_csv> <ckpt> [<ckpt> ...]
  data_dir contains <split>.csv, <split>_series.csv and <split>_series/
"""
import os
import sys

import numpy as np
import pandas as pd
import torch

from knee import LABELS, N_SLOTS, KneeNet, build_study_table, window_centres
from preprocess import MAX_SLICES, SIZE, load_series


class DicomStudyDataset(torch.utils.data.Dataset):
    """Loads every selected series of a study once; windows are cut per model setting later."""

    def __init__(self, studies, table, root, k):
        self.studies, self.table, self.root, self.k = list(studies), table, root, k

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, i):
        st = self.studies[i]
        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])
        x = np.zeros((N_SLOTS, self.k, 3, SIZE, SIZE), np.uint8)
        mask, pos = np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)
        for s, cands in enumerate(slots):
            if not cands:
                continue
            try:
                vol, _ = load_series(f"{self.root}/{st}/{cands[0]}")
            except Exception:
                continue
            n = len(vol)
            if n < 3:
                continue
            c = window_centres(n, self.k, False)
            x[s] = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])
            mask[s], pos[s] = True, c / max(n - 1, 1)
        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos)


def load_models(ckpts, device="cuda"):
    models = []
    for path in ckpts:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        # the pretrained tag after the dot is irrelevant here and unknown to older timm versions
        m = KneeNet(ck["args"]["backbone"].split(".")[0], pretrained=False, img_size=ck["args"]["res"])
        m.load_state_dict(ck["model"])
        models.append((m.to(device).to(memory_format=torch.channels_last).eval(), ck["args"]["res"]))
    return models


@torch.no_grad()
def run(data_dir, split, ckpts, k=16, studies=None, workers=4, bs=4):
    se = pd.read_csv(f"{data_dir}/{split}_series.csv")
    root = f"{data_dir}/{split}_series"
    if studies is None:
        studies = pd.read_csv(f"{data_dir}/{split}.csv").StudyInstanceUID.tolist()
    se = se[se.StudyInstanceUID.isin(set(studies))]
    n_files = {}
    for st, sr in zip(se.StudyInstanceUID, se.SeriesInstanceUID):
        d = f"{root}/{st}/{sr}"
        n_files[sr] = min(len(os.listdir(d)), MAX_SLICES) if os.path.isdir(d) else 0
    table = build_study_table(se, n_files)
    dl = torch.utils.data.DataLoader(DicomStudyDataset(studies, table, root, k), batch_size=bs,
                                     num_workers=workers, pin_memory=True)
    models = load_models(ckpts)
    out = []
    for x, mask, pos in dl:
        x, mask, pos = x.cuda(non_blocking=True), mask.cuda(), pos.cuda()
        with torch.autocast("cuda", dtype=torch.float16):
            p = torch.stack([m(x, mask, pos, res=res).float().sigmoid() for m, res in models]).mean(0)
        out.append(p.cpu())
    return pd.DataFrame(torch.cat(out).numpy(), index=pd.Index(studies, name="StudyInstanceUID"), columns=LABELS)


if __name__ == "__main__":
    data_dir, split, out_csv, ckpts = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4:]
    run(data_dir, split, ckpts).to_csv(out_csv)
    print("wrote", out_csv)

## 7. 🏆 Apex Fusion — CoAtNet + ConvNeXt Ensemble

Runs the complementary ConvNeXt reader and combines its predictions with the CoAtNet Raptor baseline using target-specific rank fusion.

**Key components:**
- `_sota_0949.csv` — preserved CoAtNet baseline predictions.
- `cnxt_v0_fold*.pt` — automatically discovers the ConvNeXt fold checkpoints.
- `_own.csv` — intermediate ConvNeXt predictions for all test studies.
- **Dependency setup** — loads available DICOM decoders and the required `timm` package.
- `TARGET_WEIGHTS` — assigns different ConvNeXt contributions to the 12 abnormalities.
- **Rank fusion** — combines percentile-ranked predictions from both readers and applies final rank normalization.
- **Fallback protection** — restores the CoAtNet submission if the fusion stage raises an exception.

**Output:** Writes the final `/kaggle/working/submission.csv` and prints `[Apex Fusion SUCCESS]` when both readers are successfully combined.

In [ ]:
# ==============================================================================
# Grandmaster Apex Fusion: SOTA CoAtNet 384px (0.949) + ConvNeXt 2.5D MIL Reader
# ==============================================================================
import os as _o_os
import sys as _o_sys
import glob as _o_glob
import time as _o_time
import shutil as _o_shutil
import subprocess as _o_sp
import pandas as _o_pd
import numpy as _o_np
import torch as _o_torch
import gc as _o_gc

_o_pub = "/kaggle/working/submission.csv"
_o_sota = "/kaggle/working/_sota_0949.csv"
_o_bak = "/kaggle/working/_sota_backup.csv"

# Ensure we have our verified 0.949 SOTA submission safely backed up
if not _o_os.path.exists(_o_sota) and _o_os.path.exists(_o_pub):
    _o_shutil.copy(_o_pub, _o_sota)
_o_shutil.copy(_o_pub, _o_bak)
print("[Apex Fusion] SOTA baseline secured at:", _o_sota, flush=True)

try:
    # 1. Clean CUDA memory from Step 1
    _o_gc.collect()
    if _o_torch.cuda.is_available():
        _o_torch.cuda.empty_cache()

    # 2. Locate ConvNeXt reader assets and competition test data
    def _o_find(name):
        for pat in ["/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"]:
            for d in sorted(_o_glob.glob(pat)):
                if _o_os.path.exists(_o_os.path.join(d, name)):
                    return d.rstrip("/")
        raise FileNotFoundError(f"{name} not found in /kaggle/input")

    _o_assets = _o_find("cnxt_v0_fold0.pt")
    _o_data = _o_find("test_series.csv")
    _o_ck = sorted(_o_glob.glob(_o_assets + "/cnxt_v0_fold*.pt"))
    print(f"[ConvNeXt Reader] Found {len(_o_ck)} checkpoints in {_o_assets}", flush=True)

    # 3. Setup environment and install decoders if wheels present
    _o_env = dict(_o_os.environ)
    _o_target = "/kaggle/working/_own_env"
    _o_sp.run([_o_sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--target", _o_target,
               "--find-links", _o_assets, "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=False)
    _o_timm = [w for pat in ["/kaggle/input/*/timm-*.whl", "/kaggle/input/*/*/timm-*.whl"]
               for w in _o_glob.glob(pat)]
    if _o_timm:
        _o_sp.run([_o_sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", _o_target, _o_timm[0]], check=False)
    _o_env["PYTHONPATH"] = _o_target + _o_os.pathsep + _o_env.get("PYTHONPATH", "")

    # Pick the GPU with highest free memory
    if _o_torch.cuda.is_available():
        _o_free = [_o_torch.cuda.mem_get_info(i)[0] for i in range(_o_torch.cuda.device_count())]
        best_gpu = max(range(len(_o_free)), key=lambda i: _o_free[i])
        _o_env["CUDA_VISIBLE_DEVICES"] = str(best_gpu)
        print(f"[ConvNeXt Reader] Selected GPU {best_gpu} with {_o_free[best_gpu] / 1e9:.1f} GB free", flush=True)

    # 4. Run ConvNeXt Reader inference
    _o_t0 = _o_time.time()
    _o_own_csv = "/kaggle/working/_own.csv"
    print("[ConvNeXt Reader] Starting inference on test set...", flush=True)
    _o_sp.run([_o_sys.executable, "/kaggle/working/own_src/infer.py", _o_data, "test", _o_own_csv] + _o_ck,
              check=True, env=_o_env)
    print(f"[ConvNeXt Reader] Finished inference in {_o_time.time() - _o_t0:.1f}s", flush=True)

    # 5. Load predictions
    sota_df = _o_pd.read_csv(_o_sota, dtype={"StudyInstanceUID": str})
    own_df = _o_pd.read_csv(_o_own_csv, dtype={"StudyInstanceUID": str})
    own_df = own_df.set_index("StudyInstanceUID").loc[sota_df.StudyInstanceUID].reset_index()

    labels = [c for c in sota_df.columns if c != "StudyInstanceUID"]
    assert own_df[labels].notna().all().all(), "ConvNeXt predictions contain NaN"
    assert (own_df[labels].nunique() > 1).all(), "ConvNeXt predictions are constant"

    # 6. Target-Specific Calibrated Rank Fusion
    # Exploit ConvNeXt strengths on Baker's Cyst (+0.036), Contusion (+0.025), Medial OA, ACL
    # while strictly relying on SOTA CoAtNet for Lateral Meniscus (+0.071), Fracture (+0.057), Lateral OA (+0.048)
    TARGET_WEIGHTS = {
        "Baker's": 0.35,           # ConvNeXt is dominant (0.976 vs 0.940)
        "Contusion": 0.30,         # ConvNeXt captures 3D bone marrow edema (0.966)
        "Medial OA": 0.25,         # ConvNeXt joint space sensitivity (0.966)
        "ACL": 0.25,               # ConvNeXt 0.965 AUC
        "Medial Meniscus": 0.20,   # Strong 2.5D sagittal slice view
        "MCL": 0.20,               # Coronal 2.5D feature extraction
        "Lateral Meniscus": 0.08,  # SOTA CoAtNet gained +0.071 (supreme)
        "Fracture": 0.08,          # SOTA CoAtNet gained +0.057 (supreme)
        "Lateral OA": 0.08,        # SOTA CoAtNet gained +0.048 (supreme)
        "PF OA": 0.08,             # SOTA CoAtNet multi-slot axial coverage
        "Effusion": 0.04,          # SOTA CoAtNet near-perfect (0.986)
        "Synovitis": 0.03,         # SOTA CoAtNet report supervision
    }

    rank_sota = sota_df[labels].rank(method="average", pct=True)
    rank_own = own_df[labels].rank(method="average", pct=True)

    fused_df = sota_df.copy()
    for col in labels:
        w = TARGET_WEIGHTS.get(col, 0.15)
        fused_df[col] = (1.0 - w) * rank_sota[col] + w * rank_own[col]

    # Final percentile rank normalization to [0, 1]
    fused_df[labels] = fused_df[labels].rank(method="average", pct=True)
    assert fused_df[labels].notna().all().all(), "Fused predictions contain NaN"
    assert (fused_df[labels].nunique() > 1).all(), "Fused predictions are constant"

    # Save to submission.csv
    fused_df.to_csv(_o_pub, index=False)
    print(f"[Apex Fusion SUCCESS] Successfully fused SOTA 0.949 + ConvNeXt 2.5D Reader into {_o_pub}!")
    print(f"[Apex Fusion SUCCESS] Applied Target Weights: {TARGET_WEIGHTS}")
    print(fused_df.head())

except Exception as e:
    print(f"[Apex Fusion EXCEPTION] {type(e).__name__}: {e}", flush=True)
    print("[Apex Fusion FALLBACK] Restoring pure SOTA 0.949 submission.csv to ensure 100% safety!", flush=True)
    _o_shutil.copy(_o_bak, _o_pub)

print("Done. Submission file size:", _o_os.path.getsize(_o_pub), "bytes")

In [ ]:
# ==== REGION 1/3: stash the base output (fallback if anything below fails) ====
import shutil as _rs_shutil
_RS_SUB = "/kaggle/working/submission.csv"; _RS_PUB = "/kaggle/working/_rs_base.csv"
_rs_shutil.copy2(_RS_SUB, _RS_PUB)
print("[region 1/3] stashed the base output", flush=True)


In [ ]:
# ==== REGION 2/3: native-resolution corpus of the studies, then the specialists ====
import glob as _rs_glob, json as _rs_json, os as _rs_os, subprocess as _rs_sp, sys as _rs_sys, time as _rs_time
import numpy as _rs_np, pandas as _rs_pd
_RS_SPEC = {'side': ['raptor_rs_side_last.pt', 'raptor_rs_side.pt'], 'regions': [{'findings': ['PF OA'], 'ckpts': ['raptor_rs_pf-fused_last.pt', 'raptor_rs_pf-fused.pt']}]}
_RS_LAB = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
_RS_W = "/kaggle/working/_rs"; _RS_C = "/tmp/rs_corpus"; _RS_CSV = "/kaggle/working/_rs_pred.csv"
_rs_ok = False


def _rs_find(name):   # weights only: never walk the competition's DICOM tree
    for d in sorted(_rs_glob.glob("/kaggle/input/*/")):
        if "competition" in d.lower() or "abnormality-detection" in d:
            continue
        hits = _rs_glob.glob(d + "**/" + name, recursive=True)
        if hits:
            return hits[0]
    raise RuntimeError(name + " not found under /kaggle/input")


def _rs_run(cmd, env=None, tail=12):
    r = _rs_sp.run(cmd, env=dict(_rs_os.environ, **(env or {})), stdout=_rs_sp.PIPE, stderr=_rs_sp.STDOUT, text=True)
    print("\n".join(r.stdout.splitlines()[-tail:]), flush=True)
    if r.returncode != 0:
        raise RuntimeError(f"{_rs_os.path.basename(cmd[1])} rc={r.returncode}")


try:
    _rs_t0 = _rs_time.time()
    _rs_os.makedirs(_RS_W, exist_ok=True); _rs_os.makedirs(_RS_C, exist_ok=True)
    open(_RS_W + "/dense512_build.py", "w").write('#!/usr/bin/env python3\n"""Dense 80-slot native-resolution cache (Kaggle CPU, one shard per kernel).\n\nSame series picker, slice picker, normalisation and 140 mm crop as dense80 (so slot j here is the\nsame slice as slot j there), at IMG=512 (0.27 mm/px ~ native). Adds per-slot DICOM geometry for the\nstage-1 localiser (maps a pixel of any slot to patient mm, so regions transfer between planes).\nOutputs in /kaggle/working: vols_s{SHARD}.npy (n,80,512,512) uint8, masks_s{SHARD}.npy,\ngeom_s{SHARD}.npy (n,80,GEOM) float32, ids_s{SHARD}.npy, meta_s{SHARD}.json.\ngeom columns: ps_row, ps_col, ipp xyz, iop row xyz, iop col xyz, y0, x0, cpx (crop in native px),\nnative rows, native cols, slice thickness, pick index, series length, has_geometry.\n"""\nimport glob\nimport json\nimport os\nimport time\nfrom multiprocessing import Pool\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport pydicom\nfrom pydicom.pixel_data_handlers.util import apply_modality_lut\n\nSHARD = 0\nN_SHARDS = 1\nIMG = 512\nGEOM = 20\nCROP_MM = 140.0\nSPAN_LO, SPAN_HI = 0.02, 0.98\nSLOTS = [("Sagittal", 1, 22), ("Sagittal", 0, 18), ("Coronal", 1, 15),\n         ("Coronal", 0, 10), ("Axial", -1, 15)]\nMAXS = sum(s[2] for s in SLOTS)\nTIME_BUDGET_S = 8.3 * 3600\nOUT = os.environ["RS_CORPUS"]\nSPLIT = os.environ.get("RS_SPLIT", "test")\ncv2.setNumThreads(1)\n\n\ndef find_root():\n    for c in ("/kaggle/input/competitions/rsna-knee-abnormality-detection",\n              "/kaggle/input/rsna-knee-abnormality-detection"):\n        if os.path.exists(c + "/train.csv"):\n            return c\n    for d, _, f in os.walk("/kaggle/input"):\n        if "train_series.csv" in f:\n            return d\n    raise SystemExit("competition data not found")\n\n\ndef order_and_meta(sdir):\n    fs = glob.glob(sdir + "/*.dcm"); recs = []; ps_list = []; info = {}\n    for f in fs:\n        try:\n            h = pydicom.dcmread(f, stop_before_pixels=True)\n            iop = getattr(h, "ImageOrientationPatient", None)\n            ipp = getattr(h, "ImagePositionPatient", None)\n            g = np.zeros(GEOM, np.float32)\n            if iop is not None and ipp is not None and len(iop) == 6:\n                r = np.array(iop[:3], float); c = np.array(iop[3:], float)\n                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))\n                g[2:5] = np.array(ipp, float); g[5:8] = r; g[8:11] = c; g[19] = 1.0\n            else:\n                pos = float(getattr(h, "InstanceNumber", 0) or 0)\n            psv = getattr(h, "PixelSpacing", None); ps = float(psv[0]) if psv is not None else 0.5\n            g[0] = ps; g[1] = float(psv[1]) if psv is not None and len(psv) > 1 else ps\n            try:\n                g[16] = float(getattr(h, "SliceThickness", 0) or 0)\n            except Exception:\n                pass\n            if not info:\n                info = {k: str(getattr(h, k, "") or "") for k in\n                        ("Laterality", "ImageLaterality", "SeriesDescription", "BodyPartExamined",\n                         "PatientPosition", "Manufacturer")}\n            ps_list.append(ps); recs.append((pos, f, ps, g))\n        except Exception:\n            recs.append((0.0, f, 0.5, np.zeros(GEOM, np.float32)))\n    recs.sort(key=lambda x: x[0])\n    med_ps = float(np.median(ps_list)) if ps_list else 0.5\n    return [(f, ps, g) for _, f, ps, g in recs], med_ps, info\n\n\ndef read_px(f):\n    d = pydicom.dcmread(f)\n    a = apply_modality_lut(d.pixel_array, d).astype(np.float32)\n    if str(getattr(d, "PhotometricInterpretation", "")) == "MONOCHROME1":\n        a = a.max() - a\n    return a\n\n\ndef mm_crop_resize(a, ps):\n    h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))\n    cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2\n    a = a[y0:y0 + cpx, x0:x0 + cpx]\n    interp = cv2.INTER_AREA if cpx >= IMG else cv2.INTER_CUBIC\n    return np.clip(cv2.resize(a, (IMG, IMG), interpolation=interp), 0, 1), (y0, x0, cpx, h, w)\n\n\ndef pick_series(rows, plane, fluid, used):\n    cands = [r for r in rows if r["Anatomical_Plane"] == plane and r["SeriesInstanceUID"] not in used]\n    if fluid in (0, 1):\n        pref = [r for r in cands if int(r.get("Fluid_Sensitive", 0) or 0) == fluid]\n        if pref:\n            return pref[0]\n    return cands[0] if cands else None\n\n\ndef fill(vol, geom, offset, picks, cache, files, med_ps):\n    arrays, spacings = [], []\n    for p in picks:\n        p = min(int(p), len(files) - 1)\n        arrays.append(cache.get(p)); ps = files[p][1]; spacings.append(ps if ps > 0 else med_ps)\n    valid = [a for a in arrays if a is not None]\n    lo, hi = (np.percentile(np.concatenate([a.ravel() for a in valid]), [2.0, 98.0])\n              if valid else (0.0, 1.0))\n    for j, (a, ps) in enumerate(zip(arrays, spacings)):\n        if offset + j >= MAXS:\n            break\n        if a is None:\n            continue\n        x, (y0, x0, cpx, h, w) = mm_crop_resize(np.clip((a - lo) / (hi - lo + 1e-6), 0, 1), ps)\n        vol[offset + j] = (x * 255).astype(np.uint8)\n        p = min(int(picks[j]), len(files) - 1)\n        g = files[p][2].copy()\n        g[11:16] = (y0, x0, cpx, h, w); g[17] = p; g[18] = len(files)\n        geom[offset + j] = g\n\n\ndef build_study(args):\n    sid, rows, sdir = args\n    vol = np.zeros((MAXS, IMG, IMG), np.uint8)\n    geom = np.zeros((MAXS, GEOM), np.float32)\n    used, offset, infos = set(), 0, []\n    try:\n        for plane, fluid, count in SLOTS:\n            rec = pick_series(rows, plane, fluid, used)\n            if rec is None:\n                offset += count; continue\n            used.add(rec["SeriesInstanceUID"])\n            files, med_ps, info = order_and_meta(f"{sdir}/{sid}/{rec[\'SeriesInstanceUID\']}")\n            info.update(slot=offset, plane=plane, fluid=fluid, n=len(files),\n                        series=rec["SeriesInstanceUID"])\n            infos.append(info)\n            if not files:\n                offset += count; continue\n            n = len(files)\n            lo = int(n * SPAN_LO); hi = max(int(n * SPAN_HI) - 1, lo)\n            picks = (np.linspace(lo, hi, count).round().astype(int) if n > 1\n                     else np.zeros(count, dtype=int))\n            cache = {}\n            for p in sorted(set(picks.tolist())):\n                p = min(int(p), n - 1)\n                try:\n                    cache[p] = read_px(files[p][0])\n                except Exception:\n                    cache[p] = None\n            fill(vol, geom, offset, picks, cache, files, med_ps)\n            offset += count\n            if offset >= MAXS:\n                break\n        err = ""\n    except Exception as e:\n        err = f"{type(e).__name__}: {e}"\n    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)\n    return vol, mask, geom, infos, err\n\n\ndef main():\n    t0 = time.time()\n    root = find_root()\n    sdir = root + f"/{SPLIT}_series"\n    if not os.path.isdir(sdir):\n        sdir = root + f"/{SPLIT}_images"\n    ids = pd.read_csv(root + f"/{SPLIT}.csv")["StudyInstanceUID"].astype(str).tolist()\n    if os.environ.get("RS_IDS"):\n        ids = json.load(open(os.environ["RS_IDS"]))\n    ser = pd.read_csv(root + f"/{SPLIT}_series.csv")\n    ser["StudyInstanceUID"] = ser["StudyInstanceUID"].astype(str)\n    ser["SeriesInstanceUID"] = ser["SeriesInstanceUID"].astype(str)\n    SER = {k: v.to_dict("records") for k, v in ser.groupby("StudyInstanceUID")}\n    b = np.linspace(0, len(ids), N_SHARDS + 1).round().astype(int)\n    mine = ids[b[SHARD]:b[SHARD + 1]]\n    print(f"root {root} | series dir {sdir} | shard {SHARD}/{N_SHARDS}: {len(mine)} studies | "\n          f"cpus {os.cpu_count()}", flush=True)\n\n    vols = np.lib.format.open_memmap(f"{OUT}/vols_s{SHARD}.npy", mode="w+", dtype=np.uint8,\n                                     shape=(len(mine), MAXS, IMG, IMG))\n    masks = np.zeros((len(mine), MAXS), np.uint8)\n    geoms = np.zeros((len(mine), MAXS, GEOM), np.float32)\n    errors, done, series_info = {}, 0, {}\n    jobs = [(sid, SER.get(sid, []), sdir) for sid in mine]\n    with Pool(os.cpu_count()) as pool:\n        for i, (vol, mask, geom, infos, err) in enumerate(pool.imap(build_study, jobs, chunksize=2)):\n            vols[i] = vol; masks[i] = mask; geoms[i] = geom; series_info[mine[i]] = infos; done = i + 1\n            if err:\n                errors[mine[i]] = err\n            if done % 50 == 0 or done == len(mine):\n                el = time.time() - t0\n                print(f"{done}/{len(mine)} | {el/60:.1f} min | eta {el/done*(len(mine)-done)/60:.1f} min "\n                      f"| mean slots {masks[:done].sum(1).mean():.1f} | errors {len(errors)}", flush=True)\n            if time.time() - t0 > TIME_BUDGET_S:\n                print("time budget hit; writing partial shard", flush=True)\n                break\n    vols.flush()\n    np.save(f"{OUT}/masks_s{SHARD}.npy", masks)\n    np.save(f"{OUT}/geom_s{SHARD}.npy", geoms)\n    json.dump(series_info, open(f"{OUT}/series_s{SHARD}.json", "w"))\n    np.save(f"{OUT}/ids_s{SHARD}.npy", np.array(mine))\n    json.dump({"shard": SHARD, "n_shards": N_SHARDS, "n": len(mine), "done": done,\n               "slots": SLOTS, "span": [SPAN_LO, SPAN_HI], "img": IMG, "crop_mm": CROP_MM,\n               "errors": errors, "empty_studies": int((masks[:done].sum(1) == 0).sum()),\n               "minutes": round((time.time() - t0) / 60, 1)},\n              open(f"{OUT}/meta_s{SHARD}.json", "w"), indent=1)\n    print("DONE", json.dumps({"done": done, "errors": len(errors)}), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')
    open(_RS_W + "/region_spec.py", "w").write('#!/usr/bin/env python3\n"""Native-resolution region specialists on the dense512 cache (PLAN_TO_TOP 2026-10-03).\n\nStage 1 (localiser, no training): the three stacks are prescribed on the joint, so the joint centre is\n    P = (x of the sagittal stack centre, y of the coronal stack centre, z of the axial stack centre)\nin patient mm (LPS), read from the per-slot DICOM geometry in geom_s*.npy. Knee side (which patient\ndirection is lateral) = DICOM Laterality, else the "side" region model (arm `--region side`), else\nsign(P.x). Canonical knee axes: A = lateral, B = anterior, C = superior.\n\nStage 2 (specialists): a region is a set of tile groups; every tile is a 2.5D window (3 adjacent slots)\nwarped from the 512 px slice to a box given in canonical mm around P, at ~native resolution, mirrored\nso that all knees look alike. `tf` sees ONE tibiofemoral compartment per sample: every study gives a\nlateral and a (mirrored) medial sample with the meniscus / OA labels of that compartment.\nTiles -> shared encoder -> tokens (+group, +slice-position) -> transformer mix -> one query per output.\n\nProtocol = family_a.py (folds_nonexpert_5, dev-29 at gold weight, holdout-29 never trained, EMA + last\nmembers, no epoch selection). Outputs: raptor_rs_<tag>.pt/_last.pt, raptor_oof_<tag>_fold<f>.npz and\nraptor_holdout_<tag>.npz (pred has all 12 columns, NaN where the region has no output),\nraptor_ft_<tag>.json; `--region side` writes raptor_side_<tag>.npz (ids, p_left) for every study.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport sys\nimport time\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\ncv2.setNumThreads(0)\nHERE = os.path.dirname(os.path.abspath(__file__))\nLAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",\n       "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nL = {f: i for i, f in enumerate(LAB)}\nWEAK5 = ["Lateral Meniscus", "Lateral OA", "PF OA", "Synovitis", "Contusion"]\nBLOCKS = {"SagF": (0, 22, "sag"), "SagNF": (22, 40, "sag"), "CorF": (40, 55, "cor"),\n          "CorNF": (55, 65, "cor"), "Ax": (65, 80, "ax")}\nIMG = 512\nN_POS = 8\nPARTNER = {"SagF": "SagNF", "CorF": "CorNF"}     # --chan seqfuse: fluid-sensitive block -> its non-fluid partner\n\n# In-plane image axes as canonical (A lateral, B anterior, C superior) unit vectors:\n# coronal: right = lateral, down = inferior | sagittal: right = posterior, down = inferior |\n# axial: right = lateral, down = posterior.  NORMAL = canonical axis that orders the slices.\nAXES = {"cor": ((1, 0, 0), (0, 0, -1)), "sag": ((0, -1, 0), (0, 0, -1)), "ax": ((1, 0, 0), (0, -1, 0))}\nNORMAL = {"cor": (0, 1, 0), "sag": (1, 0, 0), "ax": (0, 0, 1)}\n\n\ndef grp(block, hbox, vbox, out_wh, sel, n_train, n_eval):\n    """hbox/vbox: mm along the image axes (AXES) relative to P; sel: mm range along NORMAL (or None)."""\n    return dict(block=block, hbox=hbox, vbox=vbox, out=out_wh, sel=sel, n=(n_train, n_eval))\n\n\nREGIONS = {\n    # one tibiofemoral compartment; variant = (compartment sign along A, findings)\n    "tf": dict(\n        variants=[(+1, ["Lateral Meniscus", "Lateral OA"]), (-1, ["Medial Meniscus", "Medial OA"])],\n        groups=[grp("CorF", (-12, 56), (-40, 40), (256, 304), (-35, 35), 5, 8),\n                grp("CorNF", (-12, 56), (-40, 40), (256, 304), (-35, 35), 3, 5),\n                grp("SagF", (-52, 52), (-40, 40), (384, 304), (4, 52), 4, 6),\n                grp("SagNF", (-52, 52), (-40, 40), (384, 304), (4, 52), 3, 5)]),\n    "pf": dict(\n        variants=[(+1, ["PF OA"])],\n        groups=[grp("Ax", (-50, 50), (-82, 6), (368, 320), (-15, 50), 5, 8),\n                grp("SagF", (-86, 0), (-60, 40), (304, 352), (-26, 26), 4, 6),\n                grp("SagNF", (-86, 0), (-60, 40), (304, 352), (-26, 26), 3, 5)]),\n    "rec": dict(\n        variants=[(+1, ["Effusion", "Synovitis", "Baker\'s"])],\n        groups=[grp("Ax", (-70, 70), (-70, 70), (448, 448), None, 5, 8),\n                grp("SagF", (-70, 70), (-75, 45), (448, 384), None, 5, 8),\n                grp("SagNF", (-70, 70), (-75, 45), (448, 384), None, 2, 4)]),\n    "bm": dict(\n        variants=[(+1, ["Contusion", "Fracture"])],\n        groups=[grp("CorF", (-62, 62), (-55, 55), (448, 400), None, 5, 8),\n                grp("SagF", (-62, 62), (-55, 55), (448, 400), None, 5, 8),\n                grp("Ax", (-62, 62), (-62, 62), (400, 400), None, 3, 5),\n                grp("CorNF", (-62, 62), (-55, 55), (448, 400), None, 3, 5)]),\n    # knee side: coronal field incl. the fibula in PATIENT orientation (A := +x), target = left knee\n    "side": dict(\n        variants=[(+1, ["__left__"])],\n        groups=[grp("CorF", (-70, 70), (-45, 70), (224, 184), None, 3, 4),\n                grp("CorNF", (-70, 70), (-45, 70), (224, 184), None, 2, 3)]),\n}\n\n\n# ------------------------------------------------------------------ corpus + localiser\ndef _find(name, roots=(HERE, "/workspace/bundle", "/workspace/out", "/kaggle/input", "/workspace/input")):\n    for root in roots:\n        if not os.path.isdir(root):\n            continue\n        for d, _, fs in os.walk(root):\n            if name in fs:\n                return os.path.join(d, name)\n    return None\n\n\ndef slice_centres(g):\n    """(...,GEOM) -> (...,3) patient mm of the slice centres."""\n    return (g[..., 2:5] + g[..., 5:8] * (g[..., 1:2] * g[..., 15:16] / 2)\n            + g[..., 8:11] * (g[..., 0:1] * g[..., 14:15] / 2))\n\n\nclass Corpus:\n    """Memory-mapped dense512 shards + per-slot geometry; joint centre P and tag laterality per study."""\n\n    def __init__(self, n_shards=8, root=None):\n        self.n_shards, self.root, self.vols = n_shards, root, None\n        ids, masks, geom, tag = [], [], [], {}\n        for i in range(n_shards):\n            ids.append(np.load(self._p(f"ids_s{i}.npy"), allow_pickle=True).astype(str))\n            masks.append(np.load(self._p(f"masks_s{i}.npy")))\n            geom.append(np.load(self._p(f"geom_s{i}.npy")))\n            for u, infos in json.load(open(self._p(f"series_s{i}.json"))).items():\n                s = {x.get("Laterality", "")[:1].upper() for x in infos} & {"L", "R"}\n                if len(s) == 1:\n                    tag[str(u)] = 1.0 if s.pop() == "L" else 0.0\n        self.ids = np.concatenate(ids)\n        self.masks = np.concatenate(masks).astype(bool)\n        self.geom = np.concatenate(geom).astype(np.float64)\n        self.offsets = np.cumsum([0] + [len(x) for x in ids])\n        self.row = {u: i for i, u in enumerate(self.ids)}\n        self.tag_left = tag\n        cen = slice_centres(self.geom)\n        self.P = np.zeros((len(self.ids), 3))\n        for r in range(len(self.ids)):\n            allc = cen[r][self.masks[r]]\n            fall = allc.mean(0) if len(allc) else np.zeros(3)\n            for ax, blocks in enumerate((("SagF", "SagNF"), ("CorF", "CorNF"), ("Ax",))):\n                k = np.concatenate([np.arange(*BLOCKS[b][:2]) for b in blocks])\n                k = k[self.masks[r, k]]\n                self.P[r, ax] = cen[r, k, ax].mean() if len(k) else fall[ax]\n\n    def _p(self, name):\n        p = os.path.join(self.root, name) if self.root else _find(name)\n        if not p or not os.path.exists(p):\n            raise SystemExit(f"dense512 file {name} not found (DENSE_NAME=dense512 fetch_dense80.py)")\n        return p\n\n    def study(self, row):\n        if self.vols is None:\n            self.vols = [np.load(self._p(f"vols_s{i}.npy"), mmap_mode="r") for i in range(self.n_shards)]\n        s = int(np.searchsorted(self.offsets, row, side="right") - 1)\n        return self.vols[s][row - self.offsets[s]]\n\n    def left_prob(self, side_file=None):\n        """P(left knee) per study: Laterality tag > side model > sign of P.x."""\n        p = {u: (1.0 if self.P[i, 0] > 0 else 0.0) for u, i in self.row.items()}\n        src = dict.fromkeys(p, "x")\n        if side_file:\n            z = np.load(side_file, allow_pickle=True)\n            for u, v in zip(z["ids"].astype(str), z["p_left"]):\n                p[u] = float(v); src[u] = "model"\n        for u, v in self.tag_left.items():\n            p[u] = v; src[u] = "tag"\n        return p, src\n\n\ndef to_pixel(g, X):\n    """Patient mm (k,3) -> (u,v) px in the 512 slice with geometry row g."""\n    d = X - g[2:5]\n    x = d @ g[5:8] / g[1]\n    y = d @ g[8:11] / g[0]\n    return np.stack([(x - g[12]) * IMG / g[13], (y - g[11]) * IMG / g[13]], 1)\n\n\n# ------------------------------------------------------------------ data\nclass RegionTiles(Dataset):\n    """Item = (study, variant) -> {group: (n,3,H,W)} tiles + pos/valid vectors, target, weight."""\n\n    def __init__(self, corpus, region, ids, labels, weights, left, train, aug=1.0, eval_phase=0.0,\n                 side_mode=False, chan="adj"):\n        self.c, self.reg, self.train = corpus, REGIONS[region], train\n        self.chan = chan      # "seqfuse": tiles of a fluid-sensitive group = [fluid, non-fluid, difference] (see PARTNER)\n        self.labels, self.weights, self.left = labels, weights, left\n        self.items = [(u, v) for u in ids for v in range(len(self.reg["variants"]))]\n        self.aug = aug if train else 0.0\n        self.eval_phase, self.side_mode = eval_phase, side_mode\n\n    def __len__(self):\n        return len(self.items)\n\n    def _pick(self, cand, n):\n        if self.train:\n            return np.sort(np.random.choice(cand, n, replace=len(cand) < n))\n        frac = (np.linspace(0.0, 1.0, n) + self.eval_phase) % 1.0 if n > 1 else np.array([0.5])\n        return cand[np.round(frac * (len(cand) - 1)).astype(int)]\n\n    def __getitem__(self, i):\n        uid, vi = self.items[i]\n        sgn, finds = self.reg["variants"][vi]\n        row = self.c.row[uid]\n        vol, msk, geom, P = self.c.study(row), self.c.masks[row], self.c.geom[row], self.c.P[row].copy()\n        if self.side_mode:       # patient orientation; random mirror flips the target\n            lat = random.choice((1.0, -1.0)) if self.train else 1.0\n            t = self.labels.get(uid, 0.0)                 # untagged studies are only predicted\n            y = np.array([t if lat > 0 else 1.0 - t], np.float32)\n            w = np.array([self.weights.get(uid, 0.0)], np.float32)\n        else:                    # A points lateral for the lateral variant, medial for the mirrored one\n            lat = (1.0 if self.left[uid] >= 0.5 else -1.0) * sgn\n            idx = [L[f] for f in finds]\n            y, w = self.labels[uid][idx].astype(np.float32), self.weights[uid][idx].astype(np.float32)\n        E = np.array([[lat, 0, 0], [0, -1, 0], [0, 0, 1]], float)   # canonical A,B,C in patient LPS\n        rot = math.radians(random.uniform(-8, 8)) * self.aug\n        sc = 1.0 + random.uniform(-0.08, 0.08) * self.aug\n        P = P + np.random.uniform(-5, 5, 3) * self.aug\n        cr, sr = math.cos(rot), math.sin(rot)\n        out = {}\n        for gi, gdef in enumerate(self.reg["groups"]):\n            lo, hi, plane = BLOCKS[gdef["block"]]\n            n = gdef["n"][0 if self.train else 1]\n            W, H = gdef["out"]\n            eh, ev = (np.array(a, float) @ E for a in AXES[plane])\n            nrm = np.array(NORMAL[plane], float) @ E\n            valid = np.where(msk[lo:hi])[0]\n            tiles = np.zeros((n, 3, H, W), np.float32)\n            pos = np.zeros(n, np.int64)\n            ok = 0\n            if len(valid) >= 3:\n                coord = (slice_centres(geom[lo:hi]) - P) @ nrm\n                cand = valid[(valid > valid.min()) & (valid < valid.max())]\n                rng = gdef["sel"] or (coord[valid].min(), coord[valid].max())\n                inr = cand[(coord[cand] >= rng[0]) & (coord[cand] <= rng[1])]\n                if len(inr) == 0:\n                    inr = cand[np.argsort(np.abs(coord[cand] - np.mean(rng)))[:1]]\n                order = inr[np.argsort(coord[inr])]\n                ok = 1\n                (h0, h1), (v0, v1) = gdef["hbox"], gdef["vbox"]\n                hc, vc, hh, vh = (h0 + h1) / 2, (v0 + v1) / 2, (h1 - h0) / 2 * sc, (v1 - v0) / 2 * sc\n                dst = np.float32([[0, 0], [W, 0], [0, H]])\n                # --chan seqfuse: the non-fluid series of the same plane, matched in PATIENT mm (each slice is warped\n                # with its own geometry, so the two sequences are registered in the tile)\n                plo, phi = BLOCKS[PARTNER[gdef["block"]]][:2] if (self.chan == "seqfuse" and gdef["block"] in PARTNER) else (0, 0)\n                pvalid = np.where(msk[plo:phi])[0]\n                pcoord = (slice_centres(geom[plo:phi]) - P) @ nrm if len(pvalid) else None\n                for t, c in enumerate(self._pick(order, n)):\n                    c = int(c)\n                    pos[t] = int(np.clip((coord[c] - rng[0]) / max(rng[1] - rng[0], 1e-3) * N_POS, 0, N_POS - 1))\n                    corners = []\n                    for a, b in ((-hh, -vh), (hh, -vh), (-hh, vh)):\n                        a, b = a * cr - b * sr, a * sr + b * cr\n                        corners.append(P + eh * (hc + a) + ev * (vc + b))\n                    corners = np.array(corners)\n                    M = cv2.getAffineTransform(to_pixel(geom[lo + c], corners).astype(np.float32), dst)\n                    if len(pvalid):\n                        k2 = int(pvalid[np.argmin(np.abs(pcoord[pvalid] - coord[c]))])\n                        M2 = cv2.getAffineTransform(to_pixel(geom[plo + k2], corners).astype(np.float32), dst)\n                        fl = cv2.warpAffine(np.ascontiguousarray(vol[lo + c]), M, (W, H), flags=cv2.INTER_LINEAR) / 255.0\n                        nf = cv2.warpAffine(np.ascontiguousarray(vol[plo + k2]), M2, (W, H), flags=cv2.INTER_LINEAR) / 255.0\n                        tiles[t, 0], tiles[t, 1], tiles[t, 2] = fl, nf, np.clip(0.5 + (fl - nf) / 2, 0, 1)\n                        continue\n                    # neighbours ordered along the canonical normal so mirrored knees stay consistent\n                    nb = (c - 1, c, c + 1) if coord[c + 1] >= coord[c - 1] else (c + 1, c, c - 1)\n                    for ch, k in enumerate(nb):\n                        tiles[t, ch] = cv2.warpAffine(np.ascontiguousarray(vol[lo + k]), M, (W, H),\n                                                      flags=cv2.INTER_LINEAR) / 255.0\n            if self.aug:\n                tiles = np.clip(tiles * (1.0 + (random.random() - 0.5) * 0.25) + (random.random() - 0.5) * 0.06, 0, 1)\n            out[f"x{gi}"] = torch.from_numpy(tiles)\n            out[f"p{gi}"] = torch.from_numpy(pos)\n            out[f"v{gi}"] = torch.tensor(ok)\n        out["var"] = torch.tensor(vi)\n        return out, torch.from_numpy(y), torch.from_numpy(w)\n\n\ndef collate(batch):\n    x = {k: torch.stack([b[0][k] for b in batch]) for k in batch[0][0]}\n    return x, torch.stack([b[1] for b in batch]), torch.stack([b[2] for b in batch])\n\n\n# ------------------------------------------------------------------ model\nclass RegionNet(nn.Module):\n    def __init__(self, arch, n_groups, n_var, n_out, pretrained=True, d=384, heads=6, mix_layers=2,\n                 drop=0.1, grad_ckpt=False):\n        super().__init__()\n        import timm\n        self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0, global_pool="avg")\n        if grad_ckpt:\n            self.backbone.set_grad_checkpointing(True)\n        cfg = self.backbone.pretrained_cfg\n        self.register_buffer("mean", torch.tensor(cfg.get("mean", (0.485, 0.456, 0.406))).view(1, 3, 1, 1))\n        self.register_buffer("std", torch.tensor(cfg.get("std", (0.229, 0.224, 0.225))).view(1, 3, 1, 1))\n        self.proj = nn.Sequential(nn.Linear(self.backbone.num_features, d), nn.LayerNorm(d))\n        self.gemb = nn.Embedding(n_groups, d)\n        self.pemb = nn.Embedding(N_POS, d)\n        self.vemb = nn.Embedding(n_var, d)\n        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, drop, batch_first=True, norm_first=True)\n        self.mix = nn.TransformerEncoder(layer, mix_layers, enable_nested_tensor=False)\n        self.q = nn.Parameter(torch.randn(n_out, d) * 0.02)\n        self.xattn = nn.MultiheadAttention(d, heads, dropout=drop, batch_first=True)\n        self.norm = nn.LayerNorm(d)\n        self.cls_w = nn.Parameter(torch.randn(n_out, d) * 0.02)\n        self.cls_b = nn.Parameter(torch.zeros(n_out))\n        self.n_groups = n_groups\n\n    def forward(self, x):\n        toks, valid = [], []\n        for g in range(self.n_groups):\n            t = x[f"x{g}"]\n            B, N = t.shape[:2]\n            f = self.backbone((t.flatten(0, 1) - self.mean) / self.std).view(B, N, -1)\n            toks.append(self.proj(f.float()) + self.gemb.weight[g] + self.pemb(x[f"p{g}"]))\n            valid.append(x[f"v{g}"].bool()[:, None].expand(B, N))\n        h, valid = torch.cat(toks, 1), torch.cat(valid, 1)\n        valid = valid | (~valid.any(1, keepdim=True))\n        h = self.mix(h + self.vemb(x["var"])[:, None], src_key_padding_mask=~valid)\n        q = self.q[None].expand(h.shape[0], -1, -1) + self.vemb(x["var"])[:, None]\n        o, _ = self.xattn(q, h, h, key_padding_mask=~valid)\n        o = self.norm(o + q)\n        return (o * self.cls_w).sum(-1) + self.cls_b\n\n\n# ------------------------------------------------------------------ labels\ndef load_targets(a, corpus):\n    tr = pd.read_csv(a.train_csv or _find("train.csv"))\n    tr["StudyInstanceUID"] = tr["StudyInstanceUID"].astype(str)\n    gold = tr[tr[LAB].notna().all(1)].set_index("StudyInstanceUID")[LAB].astype(np.float32)\n    hold = set(json.load(open(a.holdout)))\n    lab = pd.read_parquet(a.labels) if a.labels.endswith(".parquet") else pd.read_csv(a.labels)\n    lab["StudyInstanceUID"] = lab["StudyInstanceUID"].astype(str)\n    lab = lab.set_index("StudyInstanceUID")[LAB].astype(np.float32)\n    have = set(corpus.ids)\n    labels, weights = {}, {}\n    for u, r in lab.iterrows():\n        if u in have and u not in gold.index:\n            labels[u] = r.values\n            weights[u] = 0.25 + 0.75 * np.clip(np.abs(2 * r.values - 1), 0, 1)\n    for u, r in gold.iterrows():\n        if u in have:\n            labels[u] = r.values\n            weights[u] = np.full(len(LAB), a.gold_w, np.float32)\n    holdout = [u for u in gold.index if u in hold and u in have]\n    dev = [u for u in gold.index if u not in hold and u in have]\n    folds = json.load(open(a.fold_file))["folds"]\n    nonexp = [u for u in lab.index if u in have and u not in gold.index]\n    oof = [u for u in nonexp if folds.get(u, -1) == a.fold] if a.fold >= 0 else []\n    oofset = set(oof)\n    return labels, weights, [u for u in nonexp if u not in oofset] + dev, oof, holdout\n\n\ndef auc(y, p):\n    from sklearn.metrics import roc_auc_score\n    y = (np.asarray(y) >= 0.5).astype(int)\n    return float(roc_auc_score(y, p)) if 0 < y.sum() < len(y) else float("nan")\n\n\n# ------------------------------------------------------------------ main\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--labels", required=True)\n    ap.add_argument("--region", required=True, choices=sorted(REGIONS))\n    ap.add_argument("--train_csv", default=None)\n    ap.add_argument("--fold_file", default=None)\n    ap.add_argument("--holdout", default=None)\n    ap.add_argument("--side_file", default=None, help="raptor_side_*.npz (default: search)")\n    ap.add_argument("--fold", type=int, default=0, help="-1 = train on all non-holdout studies")\n    ap.add_argument("--arch", default="convnext_small.fb_in22k_ft_in1k")\n    ap.add_argument("--epochs", type=int, default=10)\n    ap.add_argument("--bs", type=int, default=6)\n    ap.add_argument("--accum", type=int, default=2)\n    ap.add_argument("--bb_lr", type=float, default=6e-5)\n    ap.add_argument("--head_lr", type=float, default=6e-4)\n    ap.add_argument("--wd", type=float, default=0.05)\n    ap.add_argument("--warmup", type=float, default=0.06)\n    ap.add_argument("--ema", type=float, default=0.999)\n    ap.add_argument("--gold_w", type=float, default=3.0)\n    ap.add_argument("--eval_passes", type=int, default=2)\n    ap.add_argument("--eval_every", type=int, default=2)\n    ap.add_argument("--workers", type=int, default=8)\n    ap.add_argument("--grad_ckpt", action="store_true")\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--tag", default="rs")\n    ap.add_argument("--out", default=HERE)\n    ap.add_argument("--data_root", default=None)\n    ap.add_argument("--n_shards", type=int, default=8)\n    ap.add_argument("--limit", type=int, default=0)\n    ap.add_argument("--no_pretrained", action="store_true")\n    ap.add_argument("--smoke", action="store_true")\n    ap.add_argument("--dump", default=None, help="write a jpg of the first samples\' tiles and exit")\n    ap.add_argument("--chan", default="adj", choices=["adj", "seqfuse"],\n                    help="tile channels: adj = 3 adjacent slices; seqfuse = fluid-sensitive slice, the non-fluid slice of the "\n                         "same plane nearest in patient mm, and their difference (fluid-sensitive groups only)")\n    ap.add_argument("--predict", default=None,\n                    help="comma list of raptor_rs_*.pt checkpoints of ONE region (paths or names under the search "\n                         "roots): score every study of the corpus without labels, mean over the checkpoints, and write "\n                         "raptor_rspred_<tag>.npz (ids, pred (n, 12), NaN outside the region) or, for the side region, "\n                         "raptor_side_<tag>.npz (ids, p_left). Used by the test-time leg (ours_leg_rs.py)")\n    a = ap.parse_args()\n    if a.predict:\n        return predict_only(a)\n    a.fold_file = a.fold_file or _find("folds_nonexpert_5.json")\n    a.holdout = a.holdout or _find("expert_holdout.json")\n    if not a.fold_file or not a.holdout:\n        raise SystemExit("folds_nonexpert_5.json / expert_holdout.json not found")\n    if a.smoke:\n        a.epochs, a.limit, a.eval_passes, a.eval_every = 1, 24, 1, 1\n        a.workers = min(a.workers, 2)\n    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    os.makedirs(a.out, exist_ok=True)\n    reg = REGIONS[a.region]\n    side_mode = a.region == "side"\n\n    corpus = Corpus(a.n_shards, a.data_root)\n    labels, weights, train_ids, oof_ids, hold_ids = load_targets(a, corpus)\n    a.side_file = a.side_file or (None if side_mode else _find_side())\n    left, src = corpus.left_prob(a.side_file)\n    cols = [[L.get(f, -1) for f in finds] for _, finds in reg["variants"]]\n    if side_mode:                 # target = Laterality tag; only tagged studies train / score\n        tagged = corpus.tag_left\n        labels = {u: tagged[u] for u in tagged}\n        weights = dict.fromkeys(tagged, 1.0)\n        train_ids = [u for u in train_ids if u in tagged]\n        oof_ids = [u for u in oof_ids + hold_ids if u in tagged]\n        hold_ids = []\n    if a.limit:\n        train_ids = train_ids[: a.limit]; oof_ids = oof_ids[: max(8, a.limit // 2)]\n    n_src = {s: sum(v == s for v in src.values()) for s in ("tag", "model", "x")}\n    print(f"device {dev} | region {a.region} | arch {a.arch} | fold {a.fold} | train {len(train_ids)} | "\n          f"oof {len(oof_ids)} | holdout {len(hold_ids)} | side source {n_src}", flush=True)\n\n    mk = lambda ids, train, phase=0.0: RegionTiles(corpus, a.region, ids, labels, weights, left, train,\n                                                    eval_phase=phase, side_mode=side_mode, chan=a.chan)\n    if a.dump:\n        ds = mk((train_ids + oof_ids)[:4], False)\n        rows = []\n        for i in range(len(ds)):\n            x, y, _ = ds[i]\n            for g in range(len(reg["groups"])):\n                t = (x[f"x{g}"][:, 1].numpy() * 255).astype(np.uint8)\n                rows.append(cv2.resize(np.hstack(list(t[:4])), None, fx=0.6, fy=0.6))\n        w = max(r.shape[1] for r in rows)\n        cv2.imwrite(a.dump, np.vstack([np.pad(r, ((0, 4), (0, w - r.shape[1]))) for r in rows]))\n        print("dumped", a.dump, [ds.items[i] for i in range(len(ds))], [left[u] for u, _ in ds.items])\n        return 0\n\n    tl = DataLoader(mk(train_ids, True), batch_size=a.bs, shuffle=True, drop_last=True,\n                    num_workers=a.workers, collate_fn=collate, pin_memory=dev == "cuda",\n                    persistent_workers=a.workers > 0,\n                    worker_init_fn=lambda w: np.random.seed((a.seed * 1000 + w + int(time.time())) % 2**31))\n    n_out = len(cols[0])\n    model = RegionNet(a.arch, len(reg["groups"]), len(reg["variants"]), n_out,\n                      pretrained=not a.no_pretrained, grad_ckpt=a.grad_ckpt).to(dev)\n    bb = [p for n, p in model.named_parameters() if n.startswith("backbone.")]\n    hd = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]\n    opt = torch.optim.AdamW([{"params": bb, "lr": a.bb_lr}, {"params": hd, "lr": a.head_lr}], weight_decay=a.wd)\n    steps = max(1, a.epochs * (len(tl) // a.accum))\n    warm = max(1, int(a.warmup * steps))\n    sched = torch.optim.lr_scheduler.LambdaLR(\n        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / steps))))\n    if side_mode:\n        pw = torch.ones(1, len(cols), n_out, device=dev)\n    else:\n        Y = np.stack([labels[u] for u in train_ids])\n        prev = np.clip(np.stack([Y[:, c].mean(0) for c in cols]), 0.03, 0.7)        # (variants, n_out)\n        pw = torch.tensor(np.clip((1 - prev) / prev, 1, 6), dtype=torch.float32, device=dev)[None]\n    ema = {k: v.detach().float().clone() for k, v in model.state_dict().items() if v.is_floating_point()}\n\n    def fwd(x):\n        x = {k: v.to(dev, non_blocking=True) for k, v in x.items()}\n        with torch.autocast(dev, dtype=torch.bfloat16, enabled=dev == "cuda"):\n            return model(x).float(), x["var"]\n\n    def predict(ids, passes):\n        """-> (len(ids), 12) with NaN for findings outside the region (side: column 0 = p_left)."""\n        model.eval()\n        acc = 0\n        for p in range(passes):\n            ds = mk(ids, False, 0.0 if passes == 1 else p / passes)\n            dl = DataLoader(ds, batch_size=a.bs, shuffle=False, num_workers=a.workers, collate_fn=collate)\n            out = []\n            with torch.no_grad():\n                for x, _, _ in dl:\n                    out.append(torch.sigmoid(fwd(x)[0]).cpu().numpy())\n            acc = acc + np.concatenate(out)\n        acc = (acc / passes).reshape(len(ids), len(cols), n_out)\n        P = np.full((len(ids), len(LAB)), np.nan, np.float32)\n        for v, c in enumerate(cols):\n            for j, k in enumerate(c):\n                P[:, k if k >= 0 else 0] = acc[:, v, j]\n        return P\n\n    def score(P, ids):\n        if side_mode:\n            y = np.array([labels[u] for u in ids])\n            return {"auc": auc(y, P[:, 0]), "acc": float(((P[:, 0] >= 0.5) == (y >= 0.5)).mean())}\n        Yt = np.stack([labels[u] for u in ids])\n        return {LAB[k]: auc(Yt[:, k], P[:, k]) for c in cols for k in c}\n\n    def swap_in(sd):\n        cur = model.state_dict()\n        with torch.no_grad():\n            for k, v in sd.items():\n                cur[k].copy_(v.to(cur[k].dtype))\n\n    hist, t0, step = [], time.time(), 0\n    for ep in range(a.epochs):\n        model.train(); tot = 0.0; n = 0\n        opt.zero_grad(set_to_none=True)\n        for it, (x, y, w) in enumerate(tl):\n            logit, var = fwd(x)\n            y, w = y.to(dev), w.to(dev)\n            loss = (F.binary_cross_entropy_with_logits(logit, y, pos_weight=pw[0][var], reduction="none") * w).mean()\n            (loss / a.accum).backward()\n            if (it + 1) % a.accum == 0:\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)\n                opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1\n                with torch.no_grad():\n                    d = min(a.ema, (1 + step) / (10 + step))\n                    for k, v in model.state_dict().items():\n                        if k in ema:\n                            ema[k].mul_(d).add_(v.detach().float(), alpha=1 - d)\n            tot += loss.item(); n += 1\n            if it % 100 == 0:\n                print(f"  ep{ep} it{it}/{len(tl)} loss {tot / n:.4f} | {time.time() - t0:.0f}s", flush=True)\n        rec = {"ep": ep, "loss": tot / max(n, 1), "sec": round(time.time() - t0)}\n        if oof_ids and ((ep + 1) % a.eval_every == 0 or ep == a.epochs - 1):\n            live = {k: v.detach().clone() for k, v in model.state_dict().items()}\n            swap_in(ema)\n            rec["oof_ema"] = {k: round(v, 4) for k, v in score(predict(oof_ids, 1), oof_ids).items()}\n            swap_in(live)\n        hist.append(rec)\n        print(f"ep{ep} {json.dumps(rec)}", flush=True)\n\n    last = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n    meta_ck = {"arch": a.arch, "region": a.region, "spec": reg, "lab": LAB, "fold": a.fold, "seed": a.seed, "chan": a.chan}\n    torch.save({"model": last, **meta_ck}, os.path.join(a.out, f"raptor_rs_{a.tag}_last.pt"))\n    res = {"tag": a.tag, "args": vars(a), "hist": hist, "n_train": len(train_ids), "n_oof": len(oof_ids),\n           "n_holdout": len(hold_ids), "members": ["last", "ema"], "side_source": n_src}\n    all_ids = list(corpus.ids) if side_mode else []\n    po, ph, pa = [], [], []\n    for name, sd in (("last", None), ("ema", ema)):\n        if sd is not None:\n            swap_in(sd)\n            torch.save({"model": {k: v.detach().cpu() for k, v in model.state_dict().items()}, **meta_ck},\n                       os.path.join(a.out, f"raptor_rs_{a.tag}.pt"))\n        if oof_ids:\n            po.append(predict(oof_ids, a.eval_passes))\n        if hold_ids:\n            ph.append(predict(hold_ids, a.eval_passes))\n        if all_ids and not a.smoke:\n            pa.append(predict(all_ids, 1))\n    if oof_ids:\n        Po = np.mean(po, 0)\n        res["oof"] = score(Po, oof_ids)\n        if not side_mode:\n            np.savez(os.path.join(a.out, f"raptor_oof_{a.tag}_fold{a.fold}.npz"), pred=Po,\n                     truth=np.stack([labels[u] for u in oof_ids]), ids=np.array(oof_ids), fold=a.fold,\n                     pred_last=po[0], pred_ema=po[1])\n    if hold_ids:\n        Ph = np.mean(ph, 0)\n        res["holdout29"] = score(Ph, hold_ids)\n        np.savez(os.path.join(a.out, f"raptor_holdout_{a.tag}.npz"), pred=Ph,\n                 truth=np.stack([labels[u] for u in hold_ids]), ids=np.array(hold_ids),\n                 pred_last=ph[0], pred_ema=ph[1])\n    if pa:\n        p_left = np.mean(pa, 0)[:, 0]\n        np.savez(os.path.join(a.out, f"raptor_side_{a.tag}.npz"), ids=np.array(all_ids), p_left=p_left)\n        x_left = (corpus.P[:, 0] > 0)\n        res["side_all"] = {"n": len(all_ids), "frac_left": float((p_left >= 0.5).mean()),\n                           "agree_sign_x": float(((p_left >= 0.5) == x_left).mean()),\n                           "unsure": int(((p_left > 0.1) & (p_left < 0.9)).sum())}\n    json.dump(res, open(os.path.join(a.out, f"raptor_ft_{a.tag}.json"), "w"), indent=1, default=str)\n    print("RESULT " + json.dumps({k: res[k] for k in ("oof", "holdout29", "side_all") if k in res}), flush=True)\n    print(f"done in {time.time() - t0:.0f}s", flush=True)\n    return 0\n\n\ndef predict_only(a):\n    """Labels-free scoring of a whole corpus with trained checkpoints (same tiles and passes as training-time predict)."""\n    t0 = time.time()\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    os.makedirs(a.out, exist_ok=True)\n    paths = [x if os.path.exists(x) else _find(os.path.basename(x)) for x in a.predict.split(",") if x]\n    if not paths or not all(paths):\n        raise SystemExit(f"--predict: checkpoint not found among {a.predict}")\n    cks = [torch.load(x, map_location="cpu", weights_only=False) for x in paths]\n    region = cks[0]["region"]\n    assert all(c["region"] == region and c["arch"] == cks[0]["arch"] for c in cks), "one region / arch per call"\n    reg, side_mode = REGIONS[region], region == "side"\n    corpus = Corpus(a.n_shards, a.data_root)\n    ids = list(corpus.ids)\n    left, src = corpus.left_prob(None if side_mode else a.side_file)\n    cols = [[L.get(f, -1) for f in finds] for _, finds in reg["variants"]]\n    n_out = len(cols[0])\n    zero = np.zeros(len(LAB), np.float32)\n    labels, weights = ({}, {}) if side_mode else (dict.fromkeys(ids, zero), dict.fromkeys(ids, zero))\n    model = RegionNet(cks[0]["arch"], len(reg["groups"]), len(reg["variants"]), n_out, pretrained=False).to(dev)\n    model.eval()\n    n_src = {k: sum(v == k for v in src.values()) for k in ("tag", "model", "x")}\n    print(f"predict | device {dev} | region {region} | {len(paths)} checkpoint(s) | studies {len(ids)} | "\n          f"passes {a.eval_passes} | side source {n_src}", flush=True)\n    acc = 0\n    for ci, ck in enumerate(cks):\n        model.load_state_dict(ck["model"], strict=True)\n        for p in range(a.eval_passes):\n            ds = RegionTiles(corpus, region, ids, labels, weights, left, False,\n                             eval_phase=0.0 if a.eval_passes == 1 else p / a.eval_passes, side_mode=side_mode,\n                             chan=ck.get("chan", "adj"))\n            dl = DataLoader(ds, batch_size=a.bs, shuffle=False, num_workers=a.workers, collate_fn=collate)\n            out = []\n            with torch.no_grad():\n                for x, _, _ in dl:\n                    x = {k: v.to(dev, non_blocking=True) for k, v in x.items()}\n                    with torch.autocast(dev, dtype=torch.float16, enabled=dev == "cuda"):   # T4: no native bfloat16\n                        out.append(torch.sigmoid(model(x).float()).cpu().numpy())\n            acc = acc + np.concatenate(out)\n            print(f"  checkpoint {os.path.basename(paths[ci])} pass {p} | {time.time() - t0:.0f}s", flush=True)\n    acc = (acc / (len(cks) * a.eval_passes)).reshape(len(ids), len(cols), n_out)\n    P = np.full((len(ids), len(LAB)), np.nan, np.float32)\n    for v, c in enumerate(cols):\n        for j, k in enumerate(c):\n            P[:, k if k >= 0 else 0] = acc[:, v, j]\n    if side_mode:\n        np.savez(os.path.join(a.out, f"raptor_side_{a.tag}.npz"), ids=np.array(ids), p_left=P[:, 0])\n    else:\n        np.savez(os.path.join(a.out, f"raptor_rspred_{a.tag}.npz"), ids=np.array(ids), pred=P)\n    print(f"predict done | {len(ids)} studies | {time.time() - t0:.0f}s", flush=True)\n    return 0\n\n\ndef _find_side():\n    for root in ("/workspace/out", HERE, "/workspace/bundle"):\n        if os.path.isdir(root):\n            for d, _, fs in os.walk(root):\n                for f in fs:\n                    if f.startswith("raptor_side_") and f.endswith(".npz") and "smoke" not in f:\n                        return os.path.join(d, f)\n    return None\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n')
    _rs_env = {"RS_CORPUS": _RS_C, "RS_SPLIT": 'test'}
    if []:
        _rs_json.dump([], open(_RS_W + "/ids.json", "w")); _rs_env["RS_IDS"] = _RS_W + "/ids.json"
    _rs_run([_rs_sys.executable, _RS_W + "/dense512_build.py"], _rs_env, tail=4)
    _rs_meta = _rs_json.load(open(_RS_C + "/meta_s0.json"))
    print(f"[region 2/3] corpus: {_rs_meta['done']} studies, {len(_rs_meta['errors'])} errors, "
          f"{_rs_meta['empty_studies']} empty | {(_rs_time.time() - _rs_t0) / 60:.1f} min", flush=True)
    _rs_base = [_rs_sys.executable, _RS_W + "/region_spec.py", "--labels", "none", "--region", "tf", "--data_root", _RS_C,
                "--n_shards", "1", "--out", _RS_W, "--workers", "3", "--bs", "6"]
    _rs_side = []
    if _RS_SPEC["side"]:
        _rs_run(_rs_base + ["--predict", ",".join(_rs_find(c) for c in _RS_SPEC["side"]), "--tag", "side", "--eval_passes", "1"])
        _rs_side = ["--side_file", _RS_W + "/raptor_side_side.npz"]
    _rs_cols = {}
    for _rs_i, _rs_r in enumerate(_RS_SPEC["regions"]):
        _rs_run(_rs_base + ["--predict", ",".join(_rs_find(c) for c in _rs_r["ckpts"]), "--tag", f"r{_rs_i}",
                            "--eval_passes", "2"] + _rs_side)
        _rs_z = _rs_np.load(_RS_W + f"/raptor_rspred_r{_rs_i}.npz", allow_pickle=True)
        for _rs_f in _rs_r["findings"]:
            _rs_cols[_rs_f] = _rs_pd.Series(_rs_z["pred"][:, _RS_LAB.index(_rs_f)], index=_rs_z["ids"].astype(str))
    _rs_df = _rs_pd.DataFrame(_rs_cols); _rs_df.index.name = "StudyInstanceUID"; _rs_df.to_csv(_RS_CSV)
    assert _rs_df.notna().all().all(), "a specialist left NaN"
    _rs_ok = True
    print(f"[region 2/3] scored {len(_rs_df)} studies | findings {list(_rs_cols)} | {(_rs_time.time() - _rs_t0) / 60:.1f} min", flush=True)
except Exception as _rs_e:
    print(f"[region 2/3] FAILED ({type(_rs_e).__name__}: {_rs_e})", flush=True)
for _rs_p in _rs_glob.glob(_RS_W + "/*.py") + _rs_glob.glob(_RS_W + "/__pycache__/*"):
    _rs_os.remove(_rs_p)


In [ ]:
# ==== REGION 3/3: per-finding rank blend into the base; any problem -> the base stands ====
_RS_WEIGHTS = {'PF OA': 1.0}      # finding -> weight of the specialists' column (1.0 = replace the base column)
_rs_pub = _rs_pd.read_csv(_RS_PUB, dtype={"StudyInstanceUID": str})
try:
    if not _rs_ok:
        raise RuntimeError("the region leg failed")
    _rs_o = _rs_pd.read_csv(_RS_CSV, dtype={"StudyInstanceUID": str}).set_index("StudyInstanceUID").reindex(_rs_pub["StudyInstanceUID"])
    _rs_out = _rs_pub.copy(); _rs_log = []
    for _rs_f, _rs_w in _RS_WEIGHTS.items():
        _rs_c = _rs_o[_rs_f]
        if _rs_c.isna().any():
            raise RuntimeError(f"{_rs_f}: studies missing from the specialists' output")
        if len(_rs_pub) > 10 and float(_rs_c.std()) < 1e-9:
            raise RuntimeError(f"{_rs_f}: constant column")
        _rs_a = _rs_pub[_rs_f].rank(method="average", pct=True).to_numpy()
        _rs_b = _rs_c.rank(method="average", pct=True).to_numpy()
        _rs_out[_rs_f] = (1 - _rs_w) * _rs_a + _rs_w * _rs_b
        _rs_log.append(f"{_rs_f}:w{_rs_w}/corr{float(_rs_np.corrcoef(_rs_a, _rs_b)[0, 1]):.3f}")
    assert _rs_np.isfinite(_rs_out[_RS_LAB].to_numpy(float)).all()
    _rs_out.to_csv(_RS_SUB, index=False)
    print("[region 3/3] blended " + " | ".join(_rs_log), flush=True)
except Exception as _rs_e:
    _rs_shutil.copy2(_RS_PUB, _RS_SUB)
    print(f"[region 3/3] NOT blended ({type(_rs_e).__name__}: {_rs_e}); the base stands", flush=True)
_rs_final = _rs_pd.read_csv(_RS_SUB)
assert _rs_final.columns.tolist() == ["StudyInstanceUID", *_RS_LAB] and len(_rs_final) == len(_rs_pub)
print(f"[region] submission.csv ok {_rs_final.shape}", flush=True)
